### Caso nao tenha as libs instaladas no Kernel

In [10]:
# %pip install plotly pandas scikit-learn opencv-python
# %pip install --upgrade nbformat

   ---------------------------------------- 0.0/44.0 MB ? eta -:--:--
   -------- ------------------------------- 9.2/44.0 MB 47.4 MB/s eta 0:00:01
   ---------------- ----------------------- 18.1/44.0 MB 45.6 MB/s eta 0:00:01
   -------------------------- ------------- 28.8/44.0 MB 48.1 MB/s eta 0:00:01
   ----------------------------------- ---- 39.3/44.0 MB 48.0 MB/s eta 0:00:01
   ---------------------------------------- 44.0/44.0 MB 43.7 MB/s  0:00:01
   ---------------------------------------- 0.0/12.6 MB ? eta -:--:--
   -------------------------------- ------- 10.2/12.6 MB 49.1 MB/s eta 0:00:01
   ---------------------------------------- 12.6/12.6 MB 41.6 MB/s  0:00:00

  Attempting uninstall: numpy

    Found existing installation: numpy 1.26.4

   ---------------------------------------- 0/2 [numpy]
   ---------------------------------------- 0/2 [numpy]
    Uninstalling numpy-1.26.4:
   ---------------------------------------- 0/2 [numpy]
      Successfully uninstalled numpy

### Funcoes auxiliares dos experimentos complementares

In [2]:
def arrays_validos_experimento(X, y):
    """Converte entradas e alvos em arrays finitos para os modelos."""

    Xv = X.replace([np.inf, -np.inf], np.nan).fillna(0.0).to_numpy(float)
    yv = y.replace([np.inf, -np.inf], np.nan).fillna(0.0).to_numpy(float)
    return Xv, yv


def executar_multiplas_sementes(X, y, split, criar_regressor, sementes):
    """Treina a mesma MLP com varias sementes e resume a dispersao."""

    sementes = tuple(int(semente) for semente in sementes)
    if not sementes:
        raise ValueError("Informe ao menos uma semente.")
    Xv, yv = arrays_validos_experimento(X, y)
    linhas = []
    for semente in sementes:
        modelo = criar_regressor(semente)
        modelo.fit(
            Xv[split["train"]], yv[split["train"]],
            Xv[split["val"]], yv[split["val"]],
            Xv[split["test"]], yv[split["test"]],
        )
        for nome_split, chave in (("validacao", "val"), ("teste", "test")):
            metricas = avaliar_delta(
                yv[split[chave]], modelo.predict(Xv[split[chave]]),
                y.columns, "MLP", nome_split,
            )
            for linha in metricas:
                linha.update({
                    "semente": semente,
                    "melhor_epoca": modelo.best_epoch_,
                    "epoca_interrupcao": modelo.stopped_epoch_,
                    "menor_mse_validacao": modelo.best_validation_loss_,
                })
            linhas.extend(metricas)

    resultados = pd.DataFrame(linhas)
    resumo = (
        resultados.groupby(["split", "alvo_delta"], as_index=False)
        .agg(
            MAE_medio=("MAE", "mean"), MAE_desvio=("MAE", "std"),
            MAE_minimo=("MAE", "min"), MAE_maximo=("MAE", "max"),
            RMSE_medio=("RMSE", "mean"), RMSE_desvio=("RMSE", "std"),
            corr_media=("corr", "mean"),
            melhor_epoca_media=("melhor_epoca", "mean"),
        )
        .fillna({"MAE_desvio": 0.0, "RMSE_desvio": 0.0})
    )
    resumo["MAE_ic95"] = 1.96 * resumo["MAE_desvio"] / np.sqrt(len(sementes))
    return resultados, resumo


def executar_baselines_arvores(X, y, split, random_state=42):
    """Compara dois modelos de arvores com a media do treino."""

    Xv, yv = arrays_validos_experimento(X, y)
    modelos = {
        "Media treino": DummyRegressor(strategy="mean"),
        "Random Forest": RandomForestRegressor(
            n_estimators=500, min_samples_leaf=2, max_features=0.7,
            random_state=random_state, n_jobs=-1,
        ),
        "Extra Trees": ExtraTreesRegressor(
            n_estimators=500, min_samples_leaf=2, max_features=0.7,
            random_state=random_state, n_jobs=-1,
        ),
    }
    linhas = []
    for nome, modelo in modelos.items():
        modelo.fit(Xv[split["train"]], yv[split["train"]])
        for nome_split, chave in (("validacao", "val"), ("teste", "test")):
            linhas.extend(avaliar_delta(
                yv[split[chave]], modelo.predict(Xv[split[chave]]),
                y.columns, nome, nome_split,
            ))
    return pd.DataFrame(linhas), modelos


def executar_ablacao_grupos(
    X, y, split, criar_regressor, classificar_grupo, sementes=(21, 42, 84),
):
    """Remove grupos de entradas e compara o erro entre sementes pareadas."""

    sementes = tuple(int(semente) for semente in sementes)
    grupos_por_feature = pd.Series(
        {coluna: classificar_grupo(coluna) for coluna in X.columns}, name="grupo",
    )
    grupos = sorted(grupos_por_feature.unique())
    configuracoes = [("Todas as entradas", None)] + [
        (f"Sem {grupo}", grupo) for grupo in grupos
    ]
    linhas = []
    for configuracao, grupo_removido in configuracoes:
        colunas = (
            list(X.columns) if grupo_removido is None
            else grupos_por_feature[grupos_por_feature != grupo_removido].index.tolist()
        )
        if not colunas:
            continue
        Xv, yv = arrays_validos_experimento(X[colunas], y)
        for semente in sementes:
            modelo = criar_regressor(semente)
            modelo.fit(
                Xv[split["train"]], yv[split["train"]],
                Xv[split["val"]], yv[split["val"]],
                Xv[split["test"]], yv[split["test"]],
            )
            for nome_split, chave in (("validacao", "val"), ("teste", "test")):
                metricas = avaliar_delta(
                    yv[split[chave]], modelo.predict(Xv[split[chave]]),
                    y.columns, "MLP", nome_split,
                )
                for linha in metricas:
                    linha.update({
                        "configuracao": configuracao,
                        "grupo_removido": grupo_removido or "Nenhum",
                        "n_features": len(colunas),
                        "semente": semente,
                        "melhor_epoca": modelo.best_epoch_,
                    })
                linhas.extend(metricas)

    resultados = pd.DataFrame(linhas)
    referencia = (
        resultados.query("configuracao == 'Todas as entradas'")
        [["split", "alvo_delta", "semente", "MAE"]]
        .rename(columns={"MAE": "MAE_referencia"})
    )
    comparacao = resultados.merge(
        referencia, on=["split", "alvo_delta", "semente"], how="left",
    )
    comparacao["delta_MAE"] = comparacao["MAE"] - comparacao["MAE_referencia"]
    comparacao["delta_MAE_pct"] = np.where(
        comparacao["MAE_referencia"].abs() > 1e-12,
        comparacao["delta_MAE"] / comparacao["MAE_referencia"] * 100.0,
        np.nan,
    )
    return resultados, comparacao

### Import das libs

In [3]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import warnings
import json
from pathlib import Path

from IPython.display import Markdown, display
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import ExtraTreesRegressor, RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

### Analise dos logs de voo em memmap

In [4]:
lista_dfs = []
caminhos_ficheiros = []


def listar_logs_voo_memmap():
    candidatos = sorted(Path("../logs").glob("voo_teste_*/manifest.json"))
    if not candidatos:
        candidatos = sorted(Path("logs").glob("voo_teste_*/manifest.json"))
    return candidatos

def carregar_log_voo_memmap(manifest_path):
    """Carrega intervalos de voo e reconstrui trajetoria e tempo relativos."""

    manifest_path = Path(manifest_path)

    try:
        texto = manifest_path.read_text(encoding="utf-8").strip()

        if not texto:
            print(f"Ignorando manifest vazio: {manifest_path}")
            return pd.DataFrame()

        manifest = json.loads(texto)

    except JSONDecodeError as e:
        print(f"Ignorando manifest invÃ¡lido: {manifest_path}")
        print(f"Erro: {e}")
        return pd.DataFrame()

    n = int(manifest.get("num_samples", 0))
    if n <= 0:
        return pd.DataFrame()

    intervalos_path = manifest_path.parent / manifest["arrays"]["flight_intervals"]

    if not intervalos_path.exists():
        print(f"Ignorando run sem memmap: {intervalos_path}")
        return pd.DataFrame()

    intervalos = np.load(intervalos_path, mmap_mode="r")[:n]

    df = pd.DataFrame(intervalos)
    df["run_id"] = manifest_path.parent.name
    df["manifest_path"] = str(manifest_path)

    df["tempo_s"] = df["dt_s"].fillna(0).cumsum()
    df["x_rel"] = df["delta_x_m"].fillna(0).cumsum()
    df["y_rel"] = df["delta_y_m"].fillna(0).cumsum()
    df["z_rel"] = df["delta_z_m"].fillna(0).cumsum()
    df["altitude_rel"] = -df["z_rel"]

    return df

for manifest_path in listar_logs_voo_memmap():
    df_temp = carregar_log_voo_memmap(manifest_path)
    if not df_temp.empty:
        caminhos_ficheiros.append(manifest_path)
        lista_dfs.append(df_temp)

if not lista_dfs:
    display(Markdown("Nenhum log novo em memmap encontrado em `logs/voo_teste_*/manifest.json`."))

### Deslocamento acumulado relativo

In [5]:
for caminho, df in zip(caminhos_ficheiros, lista_dfs):
    timestamp = caminho.parent.name.replace("voo_teste_", "")
    fig_3d = go.Figure()
    fig_3d.add_trace(go.Scatter3d(x=df["x_rel"], y=df["y_rel"], z=df["altitude_rel"], mode="lines", line=dict(color="royalblue", width=4), name="Deslocamento acumulado"))
    fig_3d.add_trace(go.Scatter3d(x=[0.0], y=[0.0], z=[0.0], mode="markers", marker=dict(color="green", size=6), name="Origem relativa"))
    fig_3d.add_trace(go.Scatter3d(x=[df["x_rel"].iloc[-1]], y=[df["y_rel"].iloc[-1]], z=[df["altitude_rel"].iloc[-1]], mode="markers", marker=dict(color="red", size=6, symbol="x"), name="Fim relativo"))
    fig_3d.update_layout(title=f"Deslocamento acumulado por deltas - Run: {timestamp}", scene=dict(xaxis_title="Delta X acumulado (m)", yaxis_title="Delta Y acumulado (m)", zaxis_title="Delta altitude acumulada (m)", camera=dict(eye=dict(x=1.5, y=1.5, z=0.5))), legend=dict(x=0, y=1), margin=dict(l=0, r=0, b=0, t=40))
    fig_3d.show()

### Variacao das velocidades angulares

In [6]:
for caminho, df in zip(caminhos_ficheiros, lista_dfs):
    timestamp = caminho.parent.name.replace("voo_teste_", "")
    fig_2d = go.Figure()
    fig_2d.add_trace(go.Scatter(x=df["tempo_s"], y=df["delta_roll_speed_rad_s"], mode="lines", name="Delta roll speed", opacity=0.7))
    fig_2d.add_trace(go.Scatter(x=df["tempo_s"], y=df["delta_pitch_speed_rad_s"], mode="lines", name="Delta pitch speed", opacity=0.7))
    fig_2d.add_trace(go.Scatter(x=df["tempo_s"], y=df["delta_yaw_speed_rad_s"], mode="lines", name="Delta yaw speed", opacity=0.7))
    fig_2d.update_layout(title=f"Variacao das velocidades angulares - Run: {timestamp}", xaxis_title="Tempo acumulado por intervalos (s)", yaxis_title="Delta velocidade angular (rad/s)", template="plotly_white", hovermode="x unified")
    fig_2d.show()

### Analise dos intervalos depth/flow em memmap

Esta analise usa as novas runs em `datasets/depth_ground_truth/run_*/manifest.json`. Cada amostra representa a variacao entre duas atualizacoes consecutivas da logica de proximidade visual por optical flow, a mesma logica que gera as flechas desenhadas na deteccao de obstaculos.

O foco deixa de ser o estado absoluto do drone em um frame isolado e passa a ser o deslocamento sincronizado do intervalo: deltas de posicao, atitude, IMU, comandos reativos, flow e depth ground truth. O objetivo e deixar os dados menos dependentes da origem da simulacao e mais genericos para treino e analise.

In [11]:
import sys


def localizar_raiz_projeto_memmap(nome_dataset="datasets"):
    """Localiza a raiz do projeto a partir do diretorio do notebook."""

    candidatos = [
        Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent,
        Path.home() / "TCC_Drone",
    ]
    for candidato in candidatos:
        datasets_dir = candidato / nome_dataset
        if (datasets_dir / "spatial_mapping").exists() or (datasets_dir / "depth_ground_truth").exists():
            return candidato
    return Path.cwd()


raiz_dados_depth = localizar_raiz_projeto_memmap()
diretorio_modulos_analise = raiz_dados_depth / "estudos_e_analises"
if str(diretorio_modulos_analise) not in sys.path:
    sys.path.insert(0, str(diretorio_modulos_analise))
from depth_analysis_data import list_analysis_run_dirs, load_all_analysis_runs, load_analysis_run


def listar_runs_depth_memmap(base_dir):
    return list_analysis_run_dirs(Path(base_dir))


def carregar_manifesto_memmap(run_dir):
    with open(run_dir / "manifest.json", encoding="utf-8") as fp:
        return json.load(fp)


def abrir_array_memmap(run_dir, manifesto, chave):
    return np.load(run_dir / manifesto["arrays"][chave], mmap_mode="r")


def indices_intervalos_sincronizados(intervalos, max_age_s=0.08, max_interval_s=0.5):
    """Seleciona intervalos RGB/depth com tempos validos e sincronizados."""

    if intervalos.empty:
        return np.array([], dtype=int)
    dt_rgb = pd.to_numeric(intervalos["dt_s"], errors="coerce")
    dt_depth = pd.to_numeric(intervalos["depth_dt_s"], errors="coerce")
    depth_age = pd.to_numeric(intervalos["depth_age_s"], errors="coerce")
    mascara = (
        dt_rgb.notna() & dt_rgb.gt(0.0) & dt_rgb.le(max_interval_s)
        & dt_depth.notna() & dt_depth.gt(0.0) & dt_depth.le(max_interval_s)
        & depth_age.notna() & depth_age.le(max_age_s)
    )
    return np.flatnonzero(mascara.to_numpy())


def carregar_run_depth_memmap(run_dir):
    """Carrega arrays depth/flow e aplica o filtro de sincronizacao."""

    manifesto = carregar_manifesto_memmap(run_dir)
    if manifesto.get("schema_version") == "spatial_mapping_v1":
        return load_analysis_run(run_dir)
    n = int(manifesto.get("num_samples", 0))
    intervalos_mm = abrir_array_memmap(run_dir, manifesto, "intervals")
    intervalos_brutos = pd.DataFrame.from_records(intervalos_mm[:n]).copy()
    indices_validos = indices_intervalos_sincronizados(intervalos_brutos)
    intervalos = intervalos_brutos.iloc[indices_validos].reset_index(drop=True)
    if not intervalos.empty:
        intervalos["run_id"] = run_dir.name
        intervalos["ordem_intervalo"] = np.arange(len(intervalos))
    return {
        "run_dir": run_dir,
        "manifesto": manifesto,
        "intervalos": intervalos,
        "image_delta_bgr": abrir_array_memmap(run_dir, manifesto, "image_delta_bgr")[indices_validos],
        "depth_delta_log": abrir_array_memmap(run_dir, manifesto, "depth_delta_log")[indices_validos],
        "depth_delta_mask": abrir_array_memmap(run_dir, manifesto, "depth_delta_mask")[indices_validos],
        "flow_vectors": abrir_array_memmap(run_dir, manifesto, "flow_vectors")[indices_validos],
    }


def carregar_todas_runs_depth_memmap(base_dir):
    """Carrega todas as runs depth validas, isolando falhas por run."""

    run_dirs = listar_runs_depth_memmap(base_dir)
    if run_dirs and carregar_manifesto_memmap(run_dirs[0]).get("schema_version") == "spatial_mapping_v1":
        return load_all_analysis_runs(str(Path(base_dir).resolve()))
    runs = []
    for run_dir in run_dirs:
        try:
            run = carregar_run_depth_memmap(run_dir)
        except Exception as exc:
            print(f"Run ignorada em {run_dir.name}: {exc}")
            continue
        if len(run["intervalos"]) > 0:
            runs.append(run)
    return runs

In [12]:
raiz_projeto = localizar_raiz_projeto_memmap()
runs_depth_memmap = carregar_todas_runs_depth_memmap(raiz_projeto)

if not runs_depth_memmap:
    display(Markdown(
        "Nenhuma run nova em memmap encontrada em `datasets/depth_ground_truth/run_*/manifest.json`. "
        "Colete uma nova run com `save_ground_truth_dataset:=true`."
    ))
else:
    depth_run = runs_depth_memmap[-1]
    depth_df = depth_run["intervalos"].copy()
    manifesto = depth_run["manifesto"]
    print(f"Run analisada: {depth_run['run_dir'].name}")
    print(f"Intervalos depth/flow validos: {len(depth_df)}")
    print(f"Schema: {manifesto.get('schema_version')}")
    colunas_resumo = [
        "dt_s", "depth_age_s", "depth_dt_s", "delta_x_m", "delta_y_m", "delta_z_m",
        "delta_roll_rad", "delta_pitch_rad", "delta_yaw_heading_rad", "flow_valid_points",
        "flow_track_retention_pct", "flow_mag_p90_px", "radial_flow_p90_px",
        "delta_depth_p10_m", "delta_depth_p50_m", "delta_depth_close_5m_pp",
    ]
    display(depth_df[[c for c in colunas_resumo if c in depth_df.columns]].describe().T)
    fig_depth_delta = go.Figure()
    for coluna, nome in [("delta_depth_p10_m", "Delta depth P10"), ("delta_depth_p50_m", "Delta depth P50"), ("delta_depth_close_5m_pp", "Delta pixels < 5 m")]:
        if coluna in depth_df.columns:
            fig_depth_delta.add_trace(go.Scatter(x=depth_df["ordem_intervalo"], y=depth_df[coluna], mode="lines+markers", name=nome))
    fig_depth_delta.update_layout(title="Variacao de profundidade/proximidade por intervalo visual", xaxis_title="Intervalo visual em ordem de coleta", yaxis_title="Delta do intervalo", template="plotly_white", hovermode="x unified")
    fig_depth_delta.show()
    fig_flow_depth = px.scatter(depth_df, x="flow_mag_p90_px", y="delta_depth_close_5m_pp", color="radial_flow_p90_px", size="flow_valid_points", hover_data=["sample_id", "dt_s", "delta_x_m", "delta_yaw_heading_rad"], title="Flow visual x variacao de ocupacao proxima", template="plotly_white")
    fig_flow_depth.show()
    ranking = depth_df.assign(impacto_proximidade=depth_df["delta_depth_close_5m_pp"].abs()).sort_values(["impacto_proximidade", "flow_mag_p90_px"], ascending=False)
    display(Markdown("**Intervalos mais informativos para inspecao/treino:**"))
    display(ranking[["run_id", "sample_id", "dt_s", "flow_valid_points", "flow_mag_p90_px", "radial_flow_p90_px", "delta_depth_p10_m", "delta_depth_p50_m", "delta_depth_close_5m_pp", "delta_x_m", "delta_y_m", "delta_yaw_heading_rad"]].head(12))

Run analisada: run_20260831_214900_679028
Intervalos depth/flow validos: 61
Schema: spatial_mapping_v1


,count,mean,std,min,25%,50%,75%,max
dt_s,61.0,1.654361,1.118855,0.596000,0.924000,1.256000,1.880000,5.312000
depth_age_s,61.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
depth_dt_s,61.0,1.654361,1.118855,0.596000,0.924000,1.256000,1.880000,5.312000
delta_x_m,61.0,-0.408500,1.716187,-6.393080,-1.249192,-0.360023,0.092416,3.409481
delta_y_m,61.0,0.390370,2.477098,-7.475980,-0.970849,0.814411,1.807130,5.474670
delta_z_m,61.0,0.022861,0.157864,-0.207268,-0.017202,0.001151,0.033247,1.065730
delta_roll_rad,61.0,0.013354,0.367275,-0.644690,-0.330825,0.009379,0.243544,0.634301
delta_pitch_rad,61.0,-0.093171,0.616483,-2.487415,-0.226114,-0.008326,0.052819,1.733818
delta_yaw_heading_rad,61.0,0.005242,0.126293,-0.223869,-0.041016,-0.003774,0.040039,0.585787
flow_valid_points,61.0,41.475410,20.156559,5.000000,28.000000,38.000000,52.000000,94.000000


**Intervalos mais informativos para inspecao/treino:**

,run_id,sample_id,dt_s,flow_valid_points,flow_mag_p90_px,radial_flow_p90_px,delta_depth_p10_m,delta_depth_p50_m,delta_depth_close_5m_pp,delta_x_m,delta_y_m,delta_yaw_heading_rad
6,run_20260831_214900_679028,7,0.728,37.0,174.302045,-7.945577,-2.280273,-10.230469,34.902959,-0.482444,0.227211,-0.017201
52,run_20260831_214900_679028,53,1.088,28.0,106.889233,24.630257,-1.530273,-3.896484,34.282559,-0.000902,-1.283330,0.074719
21,run_20260831_214900_679028,22,0.924,34.0,121.046793,-36.596661,-2.172852,-3.078125,32.056748,-0.546004,1.465983,0.009565
54,run_20260831_214900_679028,55,0.892,22.0,106.196618,53.308402,-1.384766,-3.158203,28.233240,0.092416,-1.299046,0.018548
40,run_20260831_214900_679028,41,2.508,35.0,61.302500,37.099258,-1.792969,-1.492188,27.947384,2.073737,-2.887466,0.585787
0,run_20260831_214900_679028,1,1.816,15.0,118.848183,0.364935,0.792188,3.621094,-25.308024,-0.113173,0.344698,0.056687
53,run_20260831_214900_679028,54,2.804,11.0,127.925606,-18.612375,1.208008,2.744141,-24.224480,1.052542,-7.475980,-0.043570
51,run_20260831_214900_679028,52,1.220,18.0,178.191989,-8.533722,1.120117,2.628906,-24.055077,0.460589,-3.537441,0.114285
5,run_20260831_214900_679028,6,1.420,47.0,124.171344,8.312877,1.603516,7.671875,-22.576377,-2.627922,1.260923,-0.126910
39,run_20260831_214900_679028,40,1.284,25.0,111.035863,30.661896,2.242188,0.812500,-22.420461,-1.084822,2.792315,-0.151606


### Baseline MLP com vetores de variacao

A MLP agora trabalha com features tabulares de deslocamento entre intervalos visuais: deltas de estado, IMU, comandos, estatisticas do optical flow e estatisticas da diferenca de imagem estabilizada. Os alvos tambem sao deltas de depth/proximidade, e nao profundidades absolutas.

A comparacao contra `DummyRegressor(strategy="mean")` continua sendo usada como referencia minima: a MLP so e util se aprender uma relacao melhor que prever a variacao media observada no treino.

Para medir o efeito do aumento de dados sem alterar a dificuldade da avaliacao, as comparacoes cumulativas usam desde o primeiro marco as mesmas runs de validacao e teste. Apenas as runs de treino crescem entre os marcos. A validacao fixa e usada para decisoes de modelagem; o teste fixo serve somente para a comparacao final da curva de aprendizado.

A regressao principal usa duas camadas ocultas com dropout intercalado e early stopping. A melhor epoca e escolhida exclusivamente pela validacao fixa, e os pesos dessa epoca sao restaurados antes da avaliacao final no teste.

In [21]:
class DropoutMLP(nn.Module):
    """MLP com dropout entre as camadas ocultas."""

    def __init__(self, n_features, n_targets, hidden_layers=(64, 16), dropout=0.20):
        super().__init__()
        layers = []
        in_features = n_features
        for out_features in hidden_layers:
            layers.extend([
                nn.Linear(in_features, out_features),
                nn.ReLU(),
                nn.Dropout(dropout),
            ])
            in_features = out_features
        layers.append(nn.Linear(in_features, n_targets))
        self.network = nn.Sequential(*layers)

    def forward(self, x):
        return self.network(x)


class EarlyStoppingDropoutRegressor:
    """Regressor PyTorch com normalizacao, dropout e early stopping por validacao."""

    def __init__(
        self,
        hidden_layers=(64, 16),
        dropout=0.20,
        learning_rate=1e-3,
        weight_decay=1e-4,
        batch_size=64,
        max_epochs=500,
        patience=30,
        min_delta=1e-4,
        random_state=42,
    ):
        self.hidden_layers = hidden_layers
        self.dropout = dropout
        self.learning_rate = learning_rate
        self.weight_decay = weight_decay
        self.batch_size = batch_size
        self.max_epochs = max_epochs
        self.patience = patience
        self.min_delta = min_delta
        self.random_state = random_state

    @staticmethod
    def _as_2d(values):
        array = np.asarray(values, dtype=np.float32)
        return array.reshape(-1, 1) if array.ndim == 1 else array

    @staticmethod
    def _mse(model, X_tensor, y_tensor):
        model.eval()
        with torch.no_grad():
            return float(nn.functional.mse_loss(model(X_tensor), y_tensor).item())

    def fit(self, X_train, y_train, X_val, y_val, X_test=None, y_test=None):
        """Treina e restaura os pesos da melhor epoca, escolhida somente pela validacao."""

        X_train = self._as_2d(X_train)
        X_val = self._as_2d(X_val)
        y_train = self._as_2d(y_train)
        y_val = self._as_2d(y_val)
        self.x_scaler_ = StandardScaler().fit(X_train)
        self.y_scaler_ = StandardScaler().fit(y_train)

        X_train_scaled = self.x_scaler_.transform(X_train).astype(np.float32)
        X_val_scaled = self.x_scaler_.transform(X_val).astype(np.float32)
        y_train_scaled = self.y_scaler_.transform(y_train).astype(np.float32)
        y_val_scaled = self.y_scaler_.transform(y_val).astype(np.float32)

        torch.manual_seed(self.random_state)
        np.random.seed(self.random_state)
        self.model_ = DropoutMLP(
            X_train_scaled.shape[1],
            y_train_scaled.shape[1],
            hidden_layers=self.hidden_layers,
            dropout=self.dropout,
        )
        optimizer = torch.optim.Adam(
            self.model_.parameters(),
            lr=self.learning_rate,
            weight_decay=self.weight_decay,
        )
        loss_fn = nn.MSELoss()
        generator = torch.Generator().manual_seed(self.random_state)
        train_loader = DataLoader(
            TensorDataset(
                torch.from_numpy(X_train_scaled),
                torch.from_numpy(y_train_scaled),
            ),
            batch_size=min(self.batch_size, len(X_train_scaled)),
            shuffle=True,
            generator=generator,
        )

        tensors = {
            "treino": (
                torch.from_numpy(X_train_scaled),
                torch.from_numpy(y_train_scaled),
            ),
            "validacao": (
                torch.from_numpy(X_val_scaled),
                torch.from_numpy(y_val_scaled),
            ),
        }
        if X_test is not None and y_test is not None:
            X_test_scaled = self.x_scaler_.transform(self._as_2d(X_test)).astype(np.float32)
            y_test_scaled = self.y_scaler_.transform(self._as_2d(y_test)).astype(np.float32)
            tensors["teste"] = (
                torch.from_numpy(X_test_scaled),
                torch.from_numpy(y_test_scaled),
            )

        best_loss = np.inf
        best_state = None
        epochs_without_improvement = 0
        history = []
        for epoch in range(1, self.max_epochs + 1):
            self.model_.train()
            for X_batch, y_batch in train_loader:
                optimizer.zero_grad()
                loss = loss_fn(self.model_(X_batch), y_batch)
                loss.backward()
                optimizer.step()

            epoch_losses = {
                split_name: self._mse(self.model_, *split_tensors)
                for split_name, split_tensors in tensors.items()
            }
            for split_name, mse in epoch_losses.items():
                history.append({
                    "epoca": epoch,
                    "split": split_name,
                    "mse_padronizado": mse,
                })

            validation_loss = epoch_losses["validacao"]
            if validation_loss < best_loss - self.min_delta:
                best_loss = validation_loss
                best_state = {
                    name: value.detach().cpu().clone()
                    for name, value in self.model_.state_dict().items()
                }
                self.best_epoch_ = epoch
                epochs_without_improvement = 0
            else:
                epochs_without_improvement += 1
                if epochs_without_improvement >= self.patience:
                    break

        self.model_.load_state_dict(best_state)
        self.best_validation_loss_ = best_loss
        self.stopped_epoch_ = epoch
        self.history_ = pd.DataFrame(history)
        self.n_outputs_ = y_train.shape[1]
        return self

    def predict(self, X):
        X_scaled = self.x_scaler_.transform(self._as_2d(X)).astype(np.float32)
        self.model_.eval()
        with torch.no_grad():
            pred_scaled = self.model_(torch.from_numpy(X_scaled)).numpy()
        prediction = self.y_scaler_.inverse_transform(pred_scaled)
        return prediction.ravel() if self.n_outputs_ == 1 else prediction


TARGETS_MLP_DELTA = ["delta_depth_p10_m", "delta_depth_p50_m", "delta_depth_p90_m", "delta_depth_close_2m_pp", "delta_depth_close_5m_pp", "delta_depth_close_10m_pp"]
REGRESSOR_CONFIG = {
    "hidden_layers": (64, 16),
    "dropout": 0.30,
    "max_epochs": 300,
    "patience": 50,
    "min_delta": 1e-4,
    "random_state": 42,
}


def features_image_delta(img):
    arr = np.asarray(img, dtype=np.float32)
    abs_arr = np.abs(arr)
    return {"img_delta_mean": float(arr.mean()), "img_delta_std": float(arr.std()), "img_delta_abs_mean": float(abs_arr.mean()), "img_delta_abs_p90": float(np.percentile(abs_arr, 90))}


def features_flow_vectors(vetores, valid_points):
    n = int(max(0, min(valid_points, len(vetores))))
    if n == 0:
        return {"flow_vec_rel_std": 0.0, "flow_vec_xy_mean": 0.0, "flow_vec_radial_mean": 0.0, "flow_vec_risk_mean": 0.0}
    v = np.asarray(vetores[:n], dtype=np.float32)
    return {"flow_vec_rel_std": float(v[:, :2].std()), "flow_vec_xy_mean": float(v[:, 2:4].mean()), "flow_vec_radial_mean": float(v[:, 4].mean()), "flow_vec_risk_mean": float(v[:, 5].mean())}


def montar_dataset_mlp_delta_depth(base_dir):
    """Monta features, alvos e metadados dos intervalos depth/flow validos."""

    linhas_x, linhas_y, linhas_meta = [], [], []
    for run in carregar_todas_runs_depth_memmap(base_dir):
        df = run["intervalos"].reset_index(drop=True)
        for i, row in df.iterrows():
            if any(col not in row.index or not np.isfinite(row[col]) for col in TARGETS_MLP_DELTA):
                continue
            feats = {}
            for coluna in df.select_dtypes(include=[np.number]).columns:
                if coluna in TARGETS_MLP_DELTA or coluna.startswith("delta_depth_") or coluna in {"delta_valid_px_pct", "sample_id", "ordem_intervalo"}:
                    continue
                valor = row[coluna]
                feats[coluna] = 0.0 if pd.isna(valor) else float(valor)
            precomputed = run.get("precomputed_features")
            if precomputed is not None and i < len(precomputed):
                feats.update({k: float(v) for k, v in precomputed.iloc[i].items()})
            else:
                feats.update(features_image_delta(run["image_delta_bgr"][i]))
                feats.update(features_flow_vectors(run["flow_vectors"][i], row.get("flow_valid_points", 0)))
            linhas_x.append(feats)
            linhas_y.append({alvo: float(row[alvo]) for alvo in TARGETS_MLP_DELTA})
            linhas_meta.append({"run_id": run["run_dir"].name, "sample_id": int(row.get("sample_id", i + 1)), "ordem_intervalo": int(row.get("ordem_intervalo", i))})
    return pd.DataFrame(linhas_x), pd.DataFrame(linhas_y), pd.DataFrame(linhas_meta)


RUNS_VALIDACAO_FIXAS = {"run_20260716_221545", "run_20260715_215803"}
RUNS_TESTE_FIXAS = {"run_20260715_220346", "run_20260715_220437", "run_20260716_231519"}
MARCOS_BASE_COMPARACAO = (9, 17, 25, 33, 50, 67, 85)

def marcos_comparacao(total_runs):
    """Retorna os marcos historicos e inclui sempre o total atual de runs."""

    marcos = [marco for marco in MARCOS_BASE_COMPARACAO if marco <= total_runs]
    if total_runs and total_runs not in marcos:
        marcos.append(total_runs)
    return tuple(marcos)


def recortar_marco_runs(X, y, meta_df, runs_ordenadas, marco_runs):
    """Recorta X, y e metadados para um marco cumulativo de runs."""

    runs_marco = set(runs_ordenadas[:marco_runs])
    mascara = meta_df["run_id"].isin(runs_marco).to_numpy()
    return tuple(df.loc[mascara].reset_index(drop=True) for df in (X, y, meta_df))


def separar_intervalos(meta_df):
    """Separa intervalos por runs fixas, com fallback temporal para datasets antigos."""

    n = len(meta_df)
    indices = np.arange(n)
    runs = set(meta_df["run_id"].dropna().unique()) if n else set()
    runs_fixas = RUNS_VALIDACAO_FIXAS | RUNS_TESTE_FIXAS
    if runs_fixas.issubset(runs):
        train_runs = runs - runs_fixas
        if not train_runs:
            raise ValueError("Nenhuma run restante para treino apos aplicar os splits fixos.")
        return {
            "train": indices[meta_df["run_id"].isin(train_runs).to_numpy()],
            "val": indices[meta_df["run_id"].isin(RUNS_VALIDACAO_FIXAS).to_numpy()],
            "test": indices[meta_df["run_id"].isin(RUNS_TESTE_FIXAS).to_numpy()],
            "modo": "por run_id com validacao/teste fixos",
            "train_runs": sorted(train_runs),
            "val_runs": sorted(RUNS_VALIDACAO_FIXAS),
            "test_runs": sorted(RUNS_TESTE_FIXAS),
        }
    runs_ordenadas = list(dict.fromkeys(meta_df["run_id"].dropna().astype(str)))
    if len(runs_ordenadas) >= 3:
        n_test_runs = max(1, int(round(len(runs_ordenadas) * 0.15)))
        n_val_runs = max(1, int(round(len(runs_ordenadas) * 0.15)))
        test_runs = set(runs_ordenadas[-n_test_runs:])
        val_start = len(runs_ordenadas) - n_test_runs - n_val_runs
        val_runs = set(runs_ordenadas[max(0, val_start):-n_test_runs])
        train_runs = set(runs_ordenadas) - val_runs - test_runs
        return {
            "train": indices[meta_df["run_id"].isin(train_runs).to_numpy()],
            "val": indices[meta_df["run_id"].isin(val_runs).to_numpy()],
            "test": indices[meta_df["run_id"].isin(test_runs).to_numpy()],
            "modo": "temporal por runs completas",
            "train_runs": sorted(train_runs), "val_runs": sorted(val_runs),
            "test_runs": sorted(test_runs),
        }
    n_train = max(1, int(n * 0.70))
    n_val = max(1, int(n * 0.15))
    return {
        "train": indices[:n_train],
        "val": indices[n_train:n_train + n_val],
        "test": indices[n_train + n_val:],
        "modo": "temporal por intervalos",
    }


def avaliar_delta(y_true, y_pred, targets, modelo, split):
    """Calcula MAE, RMSE e correlacao de cada alvo de profundidade."""

    linhas = []
    for i, alvo in enumerate(targets):
        real = np.asarray(y_true[:, i], dtype=float)
        pred = np.asarray(y_pred[:, i], dtype=float)
        corr = float(np.corrcoef(real, pred)[0, 1]) if len(real) > 1 and np.std(real) > 1e-9 and np.std(pred) > 1e-9 else np.nan
        linhas.append({"modelo": modelo, "split": split, "alvo_delta": alvo, "MAE": float(mean_absolute_error(real, pred)), "RMSE": float(mean_squared_error(real, pred) ** 0.5), "corr": corr})
    return linhas


def treinar_avaliar_mlp_delta_depth(X, y, meta_df):
    """Treina a MLP com dropout/early stopping e o baseline nos mesmos splits."""

    targets = list(y.columns)
    split = separar_intervalos(meta_df)
    if len(split["test"]) == 0:
        split["test"] = split["val"]
    if len(split["val"]) == 0:
        split["val"] = split["test"]
    Xv = X.replace([np.inf, -np.inf], np.nan).fillna(0.0).to_numpy(float)
    yv = y.replace([np.inf, -np.inf], np.nan).fillna(0.0).to_numpy(float)
    mlp = EarlyStoppingDropoutRegressor(**REGRESSOR_CONFIG)
    mlp.fit(
        Xv[split["train"]],
        yv[split["train"]],
        Xv[split["val"]],
        yv[split["val"]],
        Xv[split["test"]],
        yv[split["test"]],
    )
    dummy = DummyRegressor(strategy="mean")
    dummy.fit(Xv[split["train"]], yv[split["train"]])
    linhas, predicoes = [], {}
    for nome_split, idx in [("treino", split["train"]), ("validacao", split["val"]), ("teste", split["test"])]:
        pred_mlp = mlp.predict(Xv[idx])
        pred_dummy = dummy.predict(Xv[idx])
        predicoes[nome_split] = {"idx": idx, "real": yv[idx], "mlp": pred_mlp, "dummy": pred_dummy}
        linhas.extend(avaliar_delta(yv[idx], pred_mlp, targets, "MLP", nome_split))
        linhas.extend(avaliar_delta(yv[idx], pred_dummy, targets, "Media treino", nome_split))
    return mlp, dummy, split, pd.DataFrame(linhas), predicoes


raiz_mlp = localizar_raiz_projeto_memmap()
X_mlp, y_mlp, meta_mlp = montar_dataset_mlp_delta_depth(raiz_mlp)
if len(X_mlp) < 10:
    display(Markdown("Dataset insuficiente para treinar a MLP de deltas. Colete mais intervalos depth/flow em memmap."))
else:
    runs_ordenadas_mlp = list(meta_mlp["run_id"].drop_duplicates())
    resultados_por_volume = []
    artefatos_por_volume = {}
    for marco_runs in marcos_comparacao(len(runs_ordenadas_mlp)):
        if len(runs_ordenadas_mlp) < marco_runs:
            continue
        X_marco, y_marco, meta_marco = recortar_marco_runs(
            X_mlp, y_mlp, meta_mlp, runs_ordenadas_mlp, marco_runs
        )
        modelo, baseline, split, resultados, predicoes = treinar_avaliar_mlp_delta_depth(X_marco, y_marco, meta_marco)
        resultados = resultados.assign(
            marco_runs=marco_runs,
            estrategia_split="validacao_teste_fixos",
        )
        resultados_por_volume.append(resultados)
        artefatos_por_volume[marco_runs] = {"modelo": modelo, "baseline": baseline, "split": split, "predicoes": predicoes, "meta": meta_marco}

    resultados_mlp_cumulativos = pd.concat(resultados_por_volume, ignore_index=True)
    diretorio_logs_analises = raiz_mlp / "estudos_e_analises" / "logs"
    diretorio_logs_analises.mkdir(parents=True, exist_ok=True)
    caminho_resultados_mlp = diretorio_logs_analises / "comparacao_mlp_splits_fixos.csv"
    resultados_mlp_cumulativos.to_csv(caminho_resultados_mlp, index=False)
    marco_atual = max(artefatos_por_volume)
    artefato_atual = artefatos_por_volume[marco_atual]
    modelo_mlp_delta = artefato_atual["modelo"]
    baseline_media_delta = artefato_atual["baseline"]
    split_mlp = artefato_atual["split"]
    predicoes_mlp_delta = artefato_atual["predicoes"]
    resultados_mlp_delta = resultados_mlp_cumulativos[resultados_mlp_cumulativos["marco_runs"] == marco_atual].drop(columns="marco_runs")

    display(Markdown(
        f"**Comparacao MLP com splits fixos:** validacao={sorted(RUNS_VALIDACAO_FIXAS)}; "
        f"teste={sorted(RUNS_TESTE_FIXAS)}. Apenas o treino cresce entre "
        f"{list(artefatos_por_volume)} runs."
    ))
    resumo_splits = pd.DataFrame([
        {
            "marco_runs": marco,
            "intervalos": len(artefatos_por_volume[marco]["meta"]),
            "treino": len(artefatos_por_volume[marco]["split"]["train"]),
            "validacao": len(artefatos_por_volume[marco]["split"]["val"]),
            "teste": len(artefatos_por_volume[marco]["split"]["test"]),
            "treino_pct": len(artefatos_por_volume[marco]["split"]["train"]) / len(artefatos_por_volume[marco]["meta"]) * 100.0,
            "validacao_pct": len(artefatos_por_volume[marco]["split"]["val"]) / len(artefatos_por_volume[marco]["meta"]) * 100.0,
            "teste_pct": len(artefatos_por_volume[marco]["split"]["test"]) / len(artefatos_por_volume[marco]["meta"]) * 100.0,
        }
        for marco in artefatos_por_volume
    ])
    display(resumo_splits)
    resultados_teste_cumulativos = resultados_mlp_cumulativos[resultados_mlp_cumulativos["split"] == "teste"]
    display(resultados_teste_cumulativos.sort_values(["alvo_delta", "marco_runs", "modelo"]).reset_index(drop=True))
    px.line(resultados_teste_cumulativos, x="marco_runs", y="MAE", color="modelo", facet_col="alvo_delta", facet_col_wrap=3, markers=True, title=f"Curva de aprendizado no teste fixo: {list(artefatos_por_volume)} runs", template="plotly_white").show()

    loss_por_volume = resultados_mlp_cumulativos.query("modelo == 'MLP'").copy()
    loss_por_volume["MSE"] = loss_por_volume["RMSE"] ** 2
    px.line(
        loss_por_volume,
        x="marco_runs",
        y="MSE",
        color="split",
        facet_col="alvo_delta",
        facet_col_wrap=3,
        markers=True,
        title="Loss da MLP principal por volume: treino crescente, validacao e teste fixos",
        labels={"marco_runs": "Runs disponiveis", "MSE": "MSE", "split": "Conjunto"},
        template="plotly_white",
    ).show()

    X_atual, y_atual, meta_atual = recortar_marco_runs(
        X_mlp, y_mlp, meta_mlp, runs_ordenadas_mlp, marco_atual
    )
    curvas_convergencia = modelo_mlp_delta.history_.copy()
    caminho_curvas_loss = diretorio_logs_analises / "curvas_loss_mlp.csv"
    curvas_convergencia.assign(marco_runs=marco_atual).to_csv(caminho_curvas_loss, index=False)
    validacao_loss = curvas_convergencia.query("split == 'validacao'")
    melhor_linha = validacao_loss.loc[validacao_loss["mse_padronizado"].idxmin()]
    melhor_epoca = modelo_mlp_delta.best_epoch_
    fig_loss_epocas = px.line(
        curvas_convergencia,
        x="epoca",
        y="mse_padronizado",
        color="split",
        title=f"MLP com dropout e early stopping ({marco_atual} runs)",
        labels={"epoca": "Epoca", "mse_padronizado": "MSE padronizado", "split": "Conjunto"},
        template="plotly_white",
    )
    fig_loss_epocas.add_vline(
        x=melhor_epoca,
        line_dash="dash",
        annotation_text=f"early stopping: melhor epoca {melhor_epoca}",
    )
    fig_loss_epocas.show()

    loss_final = curvas_convergencia.groupby("split", as_index=False).tail(1)
    display(pd.DataFrame([{
        "marco_runs": marco_atual,
        "melhor_epoca_validacao": melhor_epoca,
        "menor_loss_validacao": float(melhor_linha["mse_padronizado"]),
        "loss_treino_final": float(loss_final.query("split == 'treino'")["mse_padronizado"].iloc[0]),
        "loss_validacao_final": float(loss_final.query("split == 'validacao'")["mse_padronizado"].iloc[0]),
        "loss_teste_final": float(loss_final.query("split == 'teste'")["mse_padronizado"].iloc[0]),
        "epoca_interrupcao": modelo_mlp_delta.stopped_epoch_,
        "dropout": modelo_mlp_delta.dropout,
        "patience": modelo_mlp_delta.patience,
    }]))

    alvo_plot = "delta_depth_close_5m_pp" if "delta_depth_close_5m_pp" in y_mlp.columns else y_mlp.columns[0]
    alvo_idx = list(y_mlp.columns).index(alvo_plot)
    pred_teste = predicoes_mlp_delta["teste"]
    meta_teste = artefato_atual["meta"].iloc[pred_teste["idx"]].reset_index(drop=True)
    fig_pred = go.Figure()
    fig_pred.add_trace(go.Scatter(x=meta_teste.index, y=pred_teste["real"][:, alvo_idx], mode="lines+markers", name="real"))
    fig_pred.add_trace(go.Scatter(x=meta_teste.index, y=pred_teste["mlp"][:, alvo_idx], mode="lines+markers", name="MLP"))
    fig_pred.add_trace(go.Scatter(x=meta_teste.index, y=pred_teste["dummy"][:, alvo_idx], mode="lines", name="media treino"))
    fig_pred.update_layout(title=f"Predicao no teste fixo para {alvo_plot} com {marco_atual} runs", xaxis_title="Intervalos de teste em ordem temporal", yaxis_title="Delta do alvo", template="plotly_white", hovermode="x unified"); fig_pred.show()
    display(Markdown("**Intervalos do teste fixo para inspecao:**"))
    display(pd.DataFrame({"run_id": meta_teste["run_id"], "sample_id": meta_teste["sample_id"], f"{alvo_plot}_real": pred_teste["real"][:, alvo_idx], f"{alvo_plot}_mlp": pred_teste["mlp"][:, alvo_idx], f"{alvo_plot}_baseline_media": pred_teste["dummy"][:, alvo_idx]}).head(15))

**Comparacao MLP com splits fixos:** validacao=['run_20260715_215803', 'run_20260716_221545']; teste=['run_20260715_220346', 'run_20260715_220437', 'run_20260716_231519']. Apenas o treino cresce entre [9, 17, 25, 33, 50, 67, 85, 102] runs.

,marco_runs,intervalos,treino,validacao,teste,treino_pct,validacao_pct,teste_pct
0,9,631,514,116,1,81.458003,18.383518,0.158479
1,17,1052,855,57,140,81.273764,5.418251,13.307985
2,25,1584,1052,108,424,66.414141,6.818182,26.767677
3,33,2483,1224,690,569,49.295207,27.788965,22.915828
4,50,4205,2592,849,764,61.640904,20.190250,18.168847
5,67,5670,3948,890,832,69.629630,15.696649,14.673721
6,85,7068,5001,1042,1025,70.755518,14.742501,14.501981
7,102,8298,6043,1158,1097,72.824777,13.955170,13.220053


,modelo,split,alvo_delta,MAE,RMSE,corr,marco_runs,estrategia_split
0,MLP,teste,delta_depth_close_10m_pp,0.560610,0.560610,NaN,9,validacao_teste_fixos
1,Media treino,teste,delta_depth_close_10m_pp,0.433633,0.433633,NaN,9,validacao_teste_fixos
2,MLP,teste,delta_depth_close_10m_pp,5.674798,8.945151,0.180030,17,validacao_teste_fixos
3,Media treino,teste,delta_depth_close_10m_pp,5.711332,8.794143,NaN,17,validacao_teste_fixos
4,MLP,teste,delta_depth_close_10m_pp,3.059471,4.900673,0.467182,25,validacao_teste_fixos
...,...,...,...,...,...,...,...,...
91,Media treino,teste,delta_depth_p90_m,4.252908,7.163873,NaN,67,validacao_teste_fixos
92,MLP,teste,delta_depth_p90_m,3.271544,5.318874,0.713310,85,validacao_teste_fixos
93,Media treino,teste,delta_depth_p90_m,4.370495,7.438738,NaN,85,validacao_teste_fixos
94,MLP,teste,delta_depth_p90_m,3.062155,4.835965,0.726067,102,validacao_teste_fixos


,marco_runs,melhor_epoca_validacao,menor_loss_validacao,loss_treino_final,loss_validacao_final,loss_teste_final,epoca_interrupcao,dropout,patience
0,102,120,0.53563,0.382153,0.552672,0.46273,170,0.3,50


**Intervalos do teste fixo para inspecao:**

,run_id,sample_id,delta_depth_close_5m_pp_real,delta_depth_close_5m_pp_mlp,delta_depth_close_5m_pp_baseline_media
0,run_20260830_222123_856026,1,-23.575786,-14.748161,0.301946
1,run_20260830_222123_856026,2,-1.077213,-0.126735,0.301946
2,run_20260830_222123_856026,3,0.793450,0.288932,0.301946
3,run_20260830_222123_856026,4,14.461987,18.080227,0.301946
4,run_20260830_222123_856026,5,-20.748098,-20.235689,0.301946
5,run_20260830_222123_856026,6,8.931541,10.452491,0.301946
6,run_20260830_222123_856026,7,-10.047582,-12.330846,0.301946
7,run_20260830_222123_856026,8,16.313233,9.551321,0.301946
8,run_20260830_222123_856026,9,10.320986,8.668209,0.301946
9,run_20260830_222123_856026,10,-7.357502,-4.704295,0.301946


### Representacao ciclica das entradas angulares

Angulos possuem dominio ciclico: valores separados por `2*pi` representam a mesma orientacao. Embora os deltas de atitude ja sejam normalizados em `[-pi, pi]` durante a coleta, a descontinuidade entre as extremidades ainda existe para uma MLP comum.

O experimento abaixo substitui cada deslocamento angular por `sin(theta)` e `cos(theta)`. A representacao atual e a ciclica usam exatamente as mesmas runs, splits fixos, semente, dropout e early stopping. Assim, qualquer diferenca observada vem apenas da forma como os angulos entram na rede. A decisao de modelagem deve considerar a validacao; o teste permanece apenas como avaliacao final.


In [22]:
FEATURES_ANGULARES_CICLICAS = (
    "delta_roll_rad",
    "delta_pitch_rad",
    "delta_yaw_heading_rad",
    "delta_yaw_attitude_rad",
    "gyro_x_integral_rad",
    "gyro_y_integral_rad",
    "gyro_z_integral_rad",
    "pan_comp_delta_rad",
)


def codificar_angulos_ciclicos(X, features_angulares=FEATURES_ANGULARES_CICLICAS):
    """Substitui cada angulo por seno e cosseno, preservando a continuidade ciclica."""

    resultado = X.copy()
    presentes = [coluna for coluna in features_angulares if coluna in resultado.columns]
    for coluna in presentes:
        angulo = pd.to_numeric(resultado[coluna], errors="coerce").to_numpy(float)
        angulo_normalizado = np.arctan2(np.sin(angulo), np.cos(angulo))
        resultado[f"{coluna}_sin"] = np.sin(angulo_normalizado)
        resultado[f"{coluna}_cos"] = np.cos(angulo_normalizado)
    return resultado.drop(columns=presentes), presentes


if (
    "X_mlp" not in globals()
    or "resultados_mlp_cumulativos" not in globals()
    or "artefatos_por_volume" not in globals()
):
    display(Markdown("Execute primeiro a celula da MLP principal."))
else:
    X_mlp_ciclico, features_angulares_presentes = codificar_angulos_ciclicos(X_mlp)
    resultados_ciclicos = []
    artefatos_ciclicos_por_volume = {}

    for marco_runs in marcos_comparacao(len(runs_ordenadas_mlp)):
        X_marco, y_marco, meta_marco = recortar_marco_runs(
            X_mlp_ciclico, y_mlp, meta_mlp, runs_ordenadas_mlp, marco_runs
        )
        modelo, baseline, split, resultados, predicoes = treinar_avaliar_mlp_delta_depth(
            X_marco, y_marco, meta_marco
        )
        resultados_ciclicos.append(
            resultados.assign(
                marco_runs=marco_runs,
                estrategia_split="validacao_teste_fixos",
                representacao_angular="Seno/cosseno",
            )
        )
        artefatos_ciclicos_por_volume[marco_runs] = {
            "modelo": modelo,
            "baseline": baseline,
            "split": split,
            "predicoes": predicoes,
            "meta": meta_marco,
            "X": X_marco,
            "y": y_marco,
        }

    resultados_mlp_ciclicos = pd.concat(resultados_ciclicos, ignore_index=True)
    resultados_angulos_brutos = (
        resultados_mlp_cumulativos
        .query("modelo == 'MLP'")
        .assign(representacao_angular="Valor angular")
    )
    resultados_angulos_ciclicos = resultados_mlp_ciclicos.query("modelo == 'MLP'")
    comparacao_representacao_angular = pd.concat(
        [resultados_angulos_brutos, resultados_angulos_ciclicos],
        ignore_index=True,
    )
    caminho_comparacao_angular = (
        raiz_mlp
        / "estudos_e_analises"
        / "logs"
        / "comparacao_representacao_angular_splits_fixos.csv"
    )
    comparacao_representacao_angular.to_csv(caminho_comparacao_angular, index=False)

    marco_angular_atual = max(artefatos_ciclicos_por_volume)
    resumo_angular_atual = (
        comparacao_representacao_angular
        .query("marco_runs == @marco_angular_atual and split in ['validacao', 'teste']")
        .pivot_table(
            index=["split", "alvo_delta"],
            columns="representacao_angular",
            values="MAE",
        )
        .reset_index()
    )
    resumo_angular_atual["variacao_ciclica_pct"] = (
        (resumo_angular_atual["Seno/cosseno"] - resumo_angular_atual["Valor angular"])
        / resumo_angular_atual["Valor angular"]
        * 100.0
    )

    display(Markdown(
        f"**Comparacao controlada:** {len(features_angulares_presentes)} entradas angulares "
        f"foram substituidas por {2 * len(features_angulares_presentes)} componentes seno/cosseno. "
        "Valores negativos em `variacao_ciclica_pct` indicam melhora."
    ))
    display(resumo_angular_atual.round(3))

    px.line(
        comparacao_representacao_angular.query("split == 'validacao'"),
        x="marco_runs",
        y="MAE",
        color="representacao_angular",
        facet_col="alvo_delta",
        facet_col_wrap=3,
        markers=True,
        title="Representacao angular na validacao fixa: valor original x seno/cosseno",
        template="plotly_white",
    ).show()
    px.line(
        comparacao_representacao_angular.query("split == 'teste'"),
        x="marco_runs",
        y="MAE",
        color="representacao_angular",
        facet_col="alvo_delta",
        facet_col_wrap=3,
        markers=True,
        title="Resultado final da representacao angular no teste fixo",
        template="plotly_white",
    ).show()

    artefato_ciclico_atual = artefatos_ciclicos_por_volume[marco_angular_atual]
    modelo_mlp_delta_explicacao = artefato_ciclico_atual["modelo"]
    X_mlp_delta_explicacao = artefato_ciclico_atual["X"]
    y_mlp_delta_explicacao = artefato_ciclico_atual["y"]
    meta_mlp_delta_explicacao = artefato_ciclico_atual["meta"]
    split_mlp_delta_explicacao = artefato_ciclico_atual["split"]
    representacao_mlp_delta_explicacao = "seno/cosseno"


**Comparacao controlada:** 8 entradas angulares foram substituidas por 16 componentes seno/cosseno. Valores negativos em `variacao_ciclica_pct` indicam melhora.

representacao_angular,split,alvo_delta,Seno/cosseno,Valor angular,variacao_ciclica_pct
0,teste,delta_depth_close_10m_pp,3.066,3.061,0.153
1,teste,delta_depth_close_2m_pp,3.457,3.493,-1.016
2,teste,delta_depth_close_5m_pp,5.280,5.255,0.481
3,teste,delta_depth_p10_m,0.364,0.359,1.345
4,teste,delta_depth_p50_m,0.810,0.801,1.078
5,teste,delta_depth_p90_m,3.068,3.062,0.192
6,validacao,delta_depth_close_10m_pp,3.348,3.327,0.618
7,validacao,delta_depth_close_2m_pp,2.897,2.939,-1.431
8,validacao,delta_depth_close_5m_pp,5.375,5.394,-0.356
9,validacao,delta_depth_p10_m,0.362,0.359,0.966


### Explicabilidade pelo gradiente do erro nas entradas

Esta etapa usa o modelo com representacao angular por seno/cosseno quando a comparacao anterior foi executada; caso contrario, usa a MLP principal. Em ambos os casos, sao explicados os pesos restaurados pelo early stopping, sem remover runs ou retreinar a rede.

Para cada alvo, o erro quadratico padronizado e retropropagado ate as entradas. O modulo do gradiente indica quanto uma pequena mudanca em cada feature afetaria o erro daquele alvo. Como as entradas foram padronizadas com os parametros do treino, as sensibilidades ficam comparaveis. As importancias sao normalizadas para somar 100% em cada alvo.

A analise principal usa apenas a validacao fixa. O teste permanece reservado para a avaliacao final e nao participa da escolha de features, transformacoes ou arquitetura.


In [23]:
def classificar_grupo_feature(nome):
    """Agrupa features apenas para resumir as atribuicoes individuais."""

    nome = nome.lower()
    if nome.startswith("img_"):
        return "Diferenca de imagem"
    if any(token in nome for token in ("avoid_", "obstacle_", "pan_comp")):
        return "Comandos reativos"
    if "flow" in nome or "point_risk" in nome:
        return "Optical flow e risco"
    if any(token in nome for token in ("gyro", "accel", "roll", "pitch", "yaw")):
        return "IMU e atitude"
    if any(token in nome for token in ("delta_x", "delta_y", "delta_z", "smooth_v", "velocity")):
        return "Movimento"
    if any(token in nome for token in ("dt_s", "depth_age", "depth_dt", "timestamp")):
        return "Sincronizacao"
    return "Outras"


def calcular_importancia_gradiente_erro(modelo, X, y, nomes_features, nomes_alvos):
    """Calcula |d erro / d entrada| no conjunto informado sem retreinar a rede."""

    Xv = (
        X.replace([np.inf, -np.inf], np.nan)
        .fillna(0.0)
        .to_numpy(dtype=np.float32)
    )
    yv = (
        y.replace([np.inf, -np.inf], np.nan)
        .fillna(0.0)
        .to_numpy(dtype=np.float32)
    )
    X_scaled = modelo.x_scaler_.transform(Xv).astype(np.float32)
    y_scaled = modelo.y_scaler_.transform(yv).astype(np.float32)
    X_tensor = torch.tensor(X_scaled, dtype=torch.float32, requires_grad=True)
    y_tensor = torch.tensor(y_scaled, dtype=torch.float32)

    rede = modelo.model_
    rede.eval()
    with torch.no_grad():
        pred_scaled = rede(X_tensor.detach()).cpu().numpy()
    pred_original = modelo.y_scaler_.inverse_transform(pred_scaled)

    linhas_globais = []
    atribuicoes_locais = {}
    for alvo_idx, alvo in enumerate(nomes_alvos):
        rede.zero_grad(set_to_none=True)
        if X_tensor.grad is not None:
            X_tensor.grad.zero_()

        pred = rede(X_tensor)
        erros_amostra = (pred[:, alvo_idx] - y_tensor[:, alvo_idx]).square()
        erros_amostra.mean().backward()
        gradientes_abs = X_tensor.grad.detach().abs().cpu().numpy().copy()

        importancia_global = gradientes_abs.mean(axis=0)
        importancia_global /= max(float(importancia_global.sum()), 1e-12)
        soma_local = gradientes_abs.sum(axis=1, keepdims=True)
        importancia_local = gradientes_abs / np.clip(soma_local, 1e-12, None)
        atribuicoes_locais[alvo] = importancia_local

        for feature, importancia in zip(nomes_features, importancia_global):
            linhas_globais.append({
                "alvo_delta": alvo,
                "feature": feature,
                "grupo": classificar_grupo_feature(feature),
                "importancia": float(importancia),
                "importancia_pct": float(importancia * 100.0),
            })

    importancia_df = pd.DataFrame(linhas_globais)
    importancia_df["ranking_no_alvo"] = (
        importancia_df.groupby("alvo_delta")["importancia"]
        .rank(method="first", ascending=False)
        .astype(int)
    )
    return importancia_df, atribuicoes_locais, pred_original


modelo_explicado = globals().get(
    "modelo_mlp_delta_explicacao", globals().get("modelo_mlp_delta")
)
X_base_explicacao = globals().get(
    "X_mlp_delta_explicacao", globals().get("X_atual")
)
y_base_explicacao = globals().get(
    "y_mlp_delta_explicacao", globals().get("y_atual")
)
meta_base_explicacao = globals().get(
    "meta_mlp_delta_explicacao", globals().get("meta_atual")
)
split_base_explicacao = globals().get(
    "split_mlp_delta_explicacao", globals().get("split_mlp")
)
representacao_explicada = globals().get(
    "representacao_mlp_delta_explicacao", "valor angular"
)

if any(
    objeto is None
    for objeto in (
        modelo_explicado,
        X_base_explicacao,
        y_base_explicacao,
        meta_base_explicacao,
        split_base_explicacao,
    )
):
    display(Markdown(
        "Execute primeiro a celula de treinamento da MLP para gerar o melhor modelo."
    ))
else:
    indices_validacao = split_base_explicacao["val"]
    X_explicacao = X_base_explicacao.iloc[indices_validacao].reset_index(drop=True)
    y_explicacao = y_base_explicacao.iloc[indices_validacao].reset_index(drop=True)
    meta_explicacao = meta_base_explicacao.iloc[indices_validacao].reset_index(drop=True)
    nomes_features = list(X_explicacao.columns)
    nomes_alvos = list(y_explicacao.columns)

    importancia_gradiente_df, importancia_local_por_alvo, pred_explicacao = (
        calcular_importancia_gradiente_erro(
            modelo_explicado,
            X_explicacao,
            y_explicacao,
            nomes_features,
            nomes_alvos,
        )
    )
    caminho_importancia = (
        raiz_mlp
        / "estudos_e_analises"
        / "logs"
        / "importancia_gradiente_erro_validacao.csv"
    )
    importancia_gradiente_df["representacao_angular"] = representacao_explicada
    importancia_gradiente_df.to_csv(caminho_importancia, index=False)

    display(Markdown(
        f"**Modelo explicado:** representacao angular por {representacao_explicada}, "
        f"pesos da epoca {modelo_explicado.best_epoch_}, "
        f"dropout desativado durante a explicacao (`eval`) e "
        f"{len(X_explicacao)} intervalos da validacao fixa. "
        "O conjunto de teste nao foi utilizado."
    ))

    importancia_exibicao_df = importancia_gradiente_df.assign(
        feature_base=lambda df: (
            df["feature"]
            .str.replace(r"_(sin|cos)$", "", regex=True)
        )
    )
    importancia_exibicao_df = (
        importancia_exibicao_df
        .groupby(
            ["alvo_delta", "feature_base", "grupo"],
            as_index=False,
        )[["importancia", "importancia_pct"]]
        .sum()
        .rename(columns={"feature_base": "feature"})
    )
    importancia_exibicao_df["ranking_no_alvo"] = (
        importancia_exibicao_df.groupby("alvo_delta")["importancia"]
        .rank(method="first", ascending=False)
        .astype(int)
    )
    caminho_importancia_agrupada = (
        raiz_mlp
        / "estudos_e_analises"
        / "logs"
        / "importancia_gradiente_erro_validacao_agrupada.csv"
    )
    importancia_exibicao_df.to_csv(caminho_importancia_agrupada, index=False)

    top_features_por_alvo = (
        importancia_exibicao_df
        .sort_values(["alvo_delta", "importancia"], ascending=[True, False])
        .groupby("alvo_delta", as_index=False)
        .head(8)
    )
    display(Markdown("**Entradas mais associadas ao erro de cada alvo:**"))
    display(
        top_features_por_alvo[
            ["alvo_delta", "ranking_no_alvo", "feature", "grupo", "importancia_pct"]
        ].round({"importancia_pct": 2})
    )

    features_heatmap = (
        importancia_exibicao_df
        .groupby("feature")["importancia"]
        .mean()
        .nlargest(20)
        .index
    )
    matriz_importancia = (
        importancia_exibicao_df[
            importancia_exibicao_df["feature"].isin(features_heatmap)
        ]
        .pivot(index="alvo_delta", columns="feature", values="importancia_pct")
        .reindex(index=nomes_alvos, columns=features_heatmap)
    )
    fig_importancia = px.imshow(
        matriz_importancia,
        aspect="auto",
        color_continuous_scale="Blues",
        text_auto=".1f",
        labels={
            "x": "Feature de entrada",
            "y": "Alvo",
            "color": "Importancia (%)",
        },
        title="Importancia pelo gradiente do erro: alvo x entrada",
    )
    fig_importancia.update_xaxes(tickangle=45)
    fig_importancia.show()

    importancia_grupos_df = (
        importancia_exibicao_df
        .groupby(["alvo_delta", "grupo"], as_index=False)["importancia_pct"]
        .sum()
    )
    fig_grupos = px.bar(
        importancia_grupos_df,
        x="grupo",
        y="importancia_pct",
        color="grupo",
        facet_col="alvo_delta",
        facet_col_wrap=3,
        title="Soma das importancias individuais por grupo de entrada",
        labels={
            "grupo": "Grupo",
            "importancia_pct": "Importancia acumulada (%)",
            "alvo_delta": "Alvo",
        },
        template="plotly_white",
    )
    fig_grupos.update_xaxes(tickangle=35)
    fig_grupos.show()

    alvo_detalhe = (
        "delta_depth_close_5m_pp"
        if "delta_depth_close_5m_pp" in nomes_alvos
        else nomes_alvos[0]
    )
    alvo_idx = nomes_alvos.index(alvo_detalhe)
    real_detalhe = y_explicacao[alvo_detalhe].to_numpy(dtype=float)
    pred_detalhe = pred_explicacao[:, alvo_idx]
    erro_abs = np.abs(pred_detalhe - real_detalhe)
    quantidade_extremos = min(12, len(erro_abs))
    indices_extremos = np.argsort(erro_abs)[-quantidade_extremos:][::-1]

    features_detalhe = (
        importancia_gradiente_df
        .query("alvo_delta == @alvo_detalhe")
        .nlargest(15, "importancia")["feature"]
        .tolist()
    )
    indices_features = [nomes_features.index(feature) for feature in features_detalhe]
    matriz_extremos = (
        importancia_local_por_alvo[alvo_detalhe][indices_extremos][:, indices_features]
        * 100.0
    )
    rotulos_extremos = [
        (
            f"{meta_explicacao.iloc[idx]['run_id']} / "
            f"{int(meta_explicacao.iloc[idx]['sample_id'])} "
            f"(erro={erro_abs[idx]:.2f})"
        )
        for idx in indices_extremos
    ]
    fig_extremos = go.Figure(go.Heatmap(
        z=matriz_extremos,
        x=features_detalhe,
        y=rotulos_extremos,
        colorscale="Reds",
        colorbar={"title": "Importancia local (%)"},
    ))
    fig_extremos.update_layout(
        title=f"Quais entradas influenciaram os maiores erros em {alvo_detalhe}",
        xaxis_title="Feature",
        yaxis_title="Intervalo de validacao",
        template="plotly_white",
    )
    fig_extremos.update_xaxes(tickangle=45)
    fig_extremos.show()

    linhas_extremos = []
    importancia_local = importancia_local_por_alvo[alvo_detalhe]
    for idx in indices_extremos:
        importancia_amostra = importancia_local[idx]
        if float(importancia_amostra.sum()) <= 1e-12:
            features_top = ["gradiente local zero", None, None]
            importancias_top = [0.0, 0.0, 0.0]
        else:
            top_indices = np.argsort(importancia_amostra)[-3:][::-1]
            features_top = [nomes_features[top_idx] for top_idx in top_indices]
            importancias_top = [
                float(importancia_amostra[top_idx] * 100.0)
                for top_idx in top_indices
            ]
        linhas_extremos.append({
            "run_id": meta_explicacao.iloc[idx]["run_id"],
            "sample_id": int(meta_explicacao.iloc[idx]["sample_id"]),
            "real": float(real_detalhe[idx]),
            "predito": float(pred_detalhe[idx]),
            "erro_absoluto": float(erro_abs[idx]),
            "feature_1": features_top[0],
            "importancia_1_pct": importancias_top[0],
            "feature_2": features_top[1],
            "importancia_2_pct": importancias_top[1],
            "feature_3": features_top[2],
            "importancia_3_pct": importancias_top[2],
        })
    extremos_gradiente_df = pd.DataFrame(linhas_extremos)
    caminho_extremos = (
        raiz_mlp
        / "estudos_e_analises"
        / "logs"
        / "maiores_erros_gradiente_validacao.csv"
    )
    extremos_gradiente_df.to_csv(caminho_extremos, index=False)
    display(Markdown(
        f"**Maiores erros de validacao em `{alvo_detalhe}` e suas tres entradas "
        "mais influentes:**"
    ))
    display(extremos_gradiente_df.round(2))


**Modelo explicado:** representacao angular por seno/cosseno, pesos da epoca 120, dropout desativado durante a explicacao (`eval`) e 1158 intervalos da validacao fixa. O conjunto de teste nao foi utilizado.

**Entradas mais associadas ao erro de cada alvo:**

,alvo_delta,ranking_no_alvo,feature,grupo,importancia_pct
27,delta_depth_close_10m_pp,1,gyro_x_integral_rad,IMU e atitude,9.09
7,delta_depth_close_10m_pp,2,delta_roll_rad,IMU e atitude,8.05
30,delta_depth_close_10m_pp,3,img_delta_abs_mean,Diferenca de imagem,6.19
32,delta_depth_close_10m_pp,4,img_delta_mean,Diferenca de imagem,6.04
33,delta_depth_close_10m_pp,5,img_delta_std,Diferenca de imagem,5.48
9,delta_depth_close_10m_pp,6,delta_y_m,Movimento,5.39
28,delta_depth_close_10m_pp,7,gyro_y_integral_rad,IMU e atitude,4.55
21,delta_depth_close_10m_pp,8,flow_track_retention_pct,Optical flow e risco,4.46
66,delta_depth_close_2m_pp,1,gyro_x_integral_rad,IMU e atitude,8.48
46,delta_depth_close_2m_pp,2,delta_roll_rad,IMU e atitude,8.17


**Maiores erros de validacao em `delta_depth_close_5m_pp` e suas tres entradas mais influentes:**

,run_id,sample_id,real,predito,erro_absoluto,feature_1,importancia_1_pct,feature_2,importancia_2_pct,feature_3,importancia_3_pct
0,run_20260830_221718_625679,7,-51.28,-19.53,31.75,img_delta_abs_mean,11.05,delta_z_m,7.94,gyro_x_integral_rad_sin,6.23
1,run_20260830_220446_624956,17,36.77,8.16,28.61,gyro_x_integral_rad_sin,10.13,img_delta_std,8.69,flow_track_retention_pct,7.78
2,run_20260830_212610_468951,36,28.47,0.57,27.90,flow_track_retention_pct,9.19,gyro_x_integral_rad_sin,8.46,delta_roll_rad_sin,7.94
3,run_20260830_212927_939518,26,33.35,7.83,25.52,gyro_x_integral_rad_sin,10.64,img_delta_mean,9.13,delta_roll_rad_sin,7.42
4,run_20260830_214854_032567,32,33.83,8.66,25.17,img_delta_mean,10.88,gyro_x_integral_rad_sin,10.52,flow_track_retention_pct,9.13
5,run_20260830_220712_573309,34,37.21,12.25,24.95,gyro_x_integral_rad_sin,11.82,img_delta_mean,11.78,img_delta_std,9.37
6,run_20260830_220712_573309,31,36.05,11.31,24.75,flow_track_retention_pct,10.13,gyro_x_integral_rad_sin,8.76,img_delta_mean,7.68
7,run_20260830_214242_339039,24,27.19,2.46,24.73,gyro_x_integral_rad_sin,10.69,img_delta_mean,9.12,delta_y_m,8.91
8,run_20260830_220712_573309,56,27.67,3.11,24.56,img_delta_std,8.05,gyro_x_integral_rad_sin,7.98,delta_roll_rad_sin,6.11
9,run_20260830_212403_525555,35,-18.52,5.34,23.87,gyro_x_integral_rad_sin,8.75,delta_z_m,8.16,flow_track_retention_pct,7.88


### Modelo em duas etapas: evento de proximidade + regressao nos eventos

Este experimento preserva a MLP multi-output anterior e adiciona uma avaliacao alternativa para o alvo `delta_depth_close_5m_pp`. A ideia e separar a pergunta em duas partes: primeiro detectar se houve uma mudanca relevante de proximidade e, so depois, estimar a magnitude/sinal dessa mudanca.

In [24]:
from sklearn.dummy import DummyClassifier
from sklearn.metrics import average_precision_score, balanced_accuracy_score, confusion_matrix, precision_recall_fscore_support
from sklearn.neural_network import MLPClassifier

EVENT_TARGET_DELTA = "delta_depth_close_5m_pp"
EVENT_THRESHOLD_PP = 0.5
EVENT_PROBA_THRESHOLD = 0.35
EVENT_TEMPORAL_LAGS = (1, 2, 3)


def signed_log1p_array(y, scale=EVENT_THRESHOLD_PP):
    y = np.asarray(y, dtype=float)
    return np.sign(y) * np.log1p(np.abs(y) / max(scale, 1e-6))


def signed_expm1_array(z, scale=EVENT_THRESHOLD_PP):
    z = np.asarray(z, dtype=float)
    return np.sign(z) * np.expm1(np.abs(z)) * max(scale, 1e-6)


def colunas_temporais_evento(X_base):
    """Seleciona as features adequadas para construir contexto temporal."""

    preferidas = [
        "dt_s", "depth_age_s", "depth_dt_s",
        "delta_x_m", "delta_y_m", "delta_z_m",
        "delta_roll_rad", "delta_pitch_rad", "delta_yaw_heading_rad",
        "flow_valid_points", "flow_track_retention_pct",
        "flow_mag_p90_px", "radial_flow_p90_px", "pan_comp_delta_rad",
        "img_delta_abs_mean", "img_delta_abs_p90",
        "flow_vec_radial_mean", "flow_vec_risk_mean", "flow_vec_rel_std",
    ]
    return [col for col in preferidas if col in X_base.columns]


def adicionar_contexto_temporal_evento(X_base, meta_base, lags=EVENT_TEMPORAL_LAGS):
    """Adiciona lags e tendencias sem misturar intervalos entre runs."""

    X_base = X_base.reset_index(drop=True).copy()
    meta_ordem = meta_base.reset_index(drop=True).copy()
    if "ordem_intervalo" not in meta_ordem.columns:
        meta_ordem["ordem_intervalo"] = np.arange(len(meta_ordem))

    colunas = colunas_temporais_evento(X_base)
    if not colunas:
        return X_base

    trabalho = pd.concat([
        meta_ordem[["run_id", "ordem_intervalo"]].reset_index(drop=True),
        X_base[colunas].reset_index(drop=True),
    ], axis=1)
    trabalho["orig_idx"] = np.arange(len(trabalho))
    trabalho = trabalho.sort_values(["run_id", "ordem_intervalo", "orig_idx"])

    novas_partes = [trabalho]
    for lag in lags:
        lagged = trabalho.groupby("run_id")[colunas].shift(lag)
        lagged.columns = [f"{col}_lag{lag}" for col in colunas]
        novas_partes.append(lagged)

    contexto = pd.concat(novas_partes, axis=1)
    for col in colunas:
        lag3 = f"{col}_lag3"
        if lag3 in contexto.columns:
            contexto[f"{col}_trend3"] = contexto[col] - contexto[lag3]

    contexto = contexto.sort_values("orig_idx")
    contexto = contexto.drop(columns=["run_id", "ordem_intervalo", "orig_idx"], errors="ignore")
    contexto = contexto.replace([np.inf, -np.inf], np.nan).fillna(0.0)
    return pd.concat([X_base, contexto.drop(columns=colunas, errors="ignore")], axis=1)


def preparar_dataset_evento_proximidade(X_base, y_base, meta_base, target=EVENT_TARGET_DELTA, threshold=EVENT_THRESHOLD_PP):
    """Cria o alvo binario de proximidade e seu contexto temporal."""

    if target not in y_base.columns:
        raise ValueError(f"Alvo {target} nao encontrado em y_base.")

    X_evento = adicionar_contexto_temporal_evento(X_base, meta_base)
    y_delta = y_base[target].reset_index(drop=True).astype(float)
    y_evento = (np.abs(y_delta) >= threshold).astype(int)
    meta_evento = meta_base.reset_index(drop=True).copy()
    meta_evento["delta_target"] = y_delta
    meta_evento["evento_proximidade"] = y_evento
    return X_evento, y_delta, y_evento, meta_evento


def criar_classificador_evento():
    """Cria o pipeline padrao do classificador de proximidade."""

    return Pipeline([
        ("x_scaler", StandardScaler()),
        ("mlp_evento", MLPClassifier(
            hidden_layer_sizes=(32, 16),
            activation="relu",
            solver="lbfgs",
            alpha=0.05,
            max_iter=2000,
            random_state=42,
        )),
    ])


def avaliar_classificacao_evento(y_true, y_pred, y_score, modelo, split):
    """Resume desempenho, matriz de confusao e average precision."""

    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average="binary", zero_division=0
    )
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    if len(np.unique(y_true)) > 1 and np.std(y_score) > 1e-9:
        avg_precision = float(average_precision_score(y_true, y_score))
    else:
        avg_precision = np.nan

    return {
        "modelo": modelo,
        "split": split,
        "event_rate_real": float(np.mean(y_true)),
        "event_rate_pred": float(np.mean(y_pred)),
        "balanced_acc": float(balanced_accuracy_score(y_true, y_pred)),
        "precision_evento": float(precision),
        "recall_evento": float(recall),
        "f1_evento": float(f1),
        "avg_precision": avg_precision,
        "tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp),
    }


def avaliar_delta_unico(y_true, y_pred, modelo, split, alvo=EVENT_TARGET_DELTA):
    """Avalia um unico alvo continuo apos o gate de evento."""

    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    corr = float(np.corrcoef(y_true, y_pred)[0, 1]) if len(y_true) > 1 and np.std(y_true) > 1e-9 and np.std(y_pred) > 1e-9 else np.nan
    return {
        "modelo": modelo,
        "split": split,
        "alvo_delta": alvo,
        "MAE": float(mean_absolute_error(y_true, y_pred)),
        "RMSE": float(mean_squared_error(y_true, y_pred) ** 0.5),
        "corr": corr,
    }



In [25]:
if "X_mlp" not in globals() or "y_mlp" not in globals() or "meta_mlp" not in globals():
    raiz_mlp = localizar_raiz_projeto_memmap()
    X_mlp, y_mlp, meta_mlp = montar_dataset_mlp_delta_depth(raiz_mlp)

if len(X_mlp) < 12 or EVENT_TARGET_DELTA not in y_mlp.columns:
    display(Markdown("Dataset insuficiente para o modelo em duas etapas de proximidade."))
else:
    runs_ordenadas_evento = list(meta_mlp["run_id"].drop_duplicates())
    resultados_classificacao_cumulativos = []
    for marco_runs in marcos_comparacao(len(runs_ordenadas_evento)):
        if len(runs_ordenadas_evento) < marco_runs:
            continue
        X_marco, y_marco, meta_marco = recortar_marco_runs(
            X_mlp, y_mlp, meta_mlp, runs_ordenadas_evento, marco_runs
        )
        X_evento_marco, _, y_evento_marco, meta_evento_marco = preparar_dataset_evento_proximidade(
            X_marco, y_marco, meta_marco
        )
        split_marco = separar_intervalos(meta_evento_marco)
        Xv_marco = X_evento_marco.replace([np.inf, -np.inf], np.nan).fillna(0.0).to_numpy(float)
        yv_marco = y_evento_marco.to_numpy(int)
        clf_marco = criar_classificador_evento()
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            clf_marco.fit(Xv_marco[split_marco["train"]], yv_marco[split_marco["train"]])
        for nome_split, idx in [("validacao", split_marco["val"]), ("teste", split_marco["test"])]:
            proba = clf_marco.predict_proba(Xv_marco[idx])[:, 1]
            pred = (proba >= EVENT_PROBA_THRESHOLD).astype(int)
            linha = avaliar_classificacao_evento(yv_marco[idx], pred, proba, "MLP evento", nome_split)
            linha["marco_runs"] = marco_runs
            linha["estrategia_split"] = "validacao_teste_fixos"
            resultados_classificacao_cumulativos.append(linha)

    resultados_classificacao_cumulativos = pd.DataFrame(resultados_classificacao_cumulativos)
    caminho_resultados_evento = diretorio_logs_analises / "comparacao_eventos_splits_fixos.csv"
    resultados_classificacao_cumulativos.to_csv(caminho_resultados_evento, index=False)
    display(Markdown("**Curva do classificador com validacao e teste fixos:**"))
    display(resultados_classificacao_cumulativos.sort_values(["split", "marco_runs"]).reset_index(drop=True))
    px.line(resultados_classificacao_cumulativos, x="marco_runs", y="balanced_acc", color="split", markers=True, title=f"Balanced accuracy com splits fixos: {sorted(resultados_classificacao_cumulativos['marco_runs'].unique())} runs", template="plotly_white").show()

    X_evento, y_delta_evento, y_evento, meta_evento = preparar_dataset_evento_proximidade(
        X_mlp, y_mlp, meta_mlp
    )
    split_evento = separar_intervalos(meta_evento)
    if len(split_evento["test"]) == 0:
        split_evento["test"] = split_evento["val"]
    if len(split_evento["val"]) == 0:
        split_evento["val"] = split_evento["test"]

    display(Markdown(
        f"**Dataset evento proximidade:** {len(X_evento)} intervalos, {X_evento.shape[1]} features "
        f"com contexto temporal, alvo `{EVENT_TARGET_DELTA}`, limiar={EVENT_THRESHOLD_PP:.2f} p.p. "
        f"Taxa de evento={float(y_evento.mean()):.1%}. Separacao: {split_evento['modo']}. "
        f"Treino={len(split_evento['train'])}, validacao={len(split_evento['val'])}, teste={len(split_evento['test'])}."
    ))

    resumo_runs_evento = meta_evento.groupby("run_id").agg(
        intervalos=("evento_proximidade", "size"),
        eventos=("evento_proximidade", "sum"),
        taxa_evento=("evento_proximidade", "mean"),
        delta_abs_p90=("delta_target", lambda s: float(np.percentile(np.abs(s), 90))),
    ).reset_index()
    display(resumo_runs_evento)

    Xv_evento = X_evento.replace([np.inf, -np.inf], np.nan).fillna(0.0).to_numpy(float)
    y_delta_v = y_delta_evento.to_numpy(float)
    y_evento_v = y_evento.to_numpy(int)

    train_idx = split_evento["train"]
    train_event_idx = train_idx[y_evento_v[train_idx] == 1]

    if len(np.unique(y_evento_v[train_idx])) < 2 or len(train_event_idx) < 5:
        display(Markdown(
            "A separacao atual nao tem exemplos suficientes de evento no treino para classificador + regressor. "
            "Diminua `EVENT_THRESHOLD_PP` ou colete mais runs com aproximacao real de obstaculos."
        ))
    else:
        clf_evento = criar_classificador_evento()
        clf_dummy = DummyClassifier(strategy="most_frequent")

        reg_evento = EarlyStoppingDropoutRegressor(**REGRESSOR_CONFIG)
        reg_dummy_evento = DummyRegressor(strategy="median")

        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            clf_evento.fit(Xv_evento[train_idx], y_evento_v[train_idx])
            val_event_idx = split_evento["val"][y_evento_v[split_evento["val"]] == 1]
            test_event_idx = split_evento["test"][y_evento_v[split_evento["test"]] == 1]
            reg_evento.fit(
                Xv_evento[train_event_idx],
                signed_log1p_array(y_delta_v[train_event_idx]),
                Xv_evento[val_event_idx],
                signed_log1p_array(y_delta_v[val_event_idx]),
                Xv_evento[test_event_idx],
                signed_log1p_array(y_delta_v[test_event_idx]),
            )

        clf_dummy.fit(Xv_evento[train_idx], y_evento_v[train_idx])
        reg_dummy_evento.fit(Xv_evento[train_event_idx], y_delta_v[train_event_idx])

        linhas_classificacao = []
        linhas_regressao = []
        predicoes_evento = {}

        for nome_split, idx in [("validacao", split_evento["val"]), ("teste", split_evento["test"] )]:
            if len(idx) == 0:
                continue

            proba_evento = clf_evento.predict_proba(Xv_evento[idx])[:, 1]
            pred_evento = (proba_evento >= EVENT_PROBA_THRESHOLD).astype(int)
            pred_evento_dummy = clf_dummy.predict(Xv_evento[idx]).astype(int)
            score_dummy = np.full(len(idx), float(np.mean(y_evento_v[train_idx])))

            linhas_classificacao.append(avaliar_classificacao_evento(
                y_evento_v[idx], pred_evento, proba_evento, "MLP evento", nome_split
            ))
            linhas_classificacao.append(avaliar_classificacao_evento(
                y_evento_v[idx], pred_evento_dummy, score_dummy, "Classe majoritaria", nome_split
            ))

            pred_delta_evento = signed_expm1_array(reg_evento.predict(Xv_evento[idx]).ravel())
            pred_delta_gate = np.where(pred_evento == 1, pred_delta_evento, 0.0)
            pred_delta_evento_real_gate = np.where(y_evento_v[idx] == 1, pred_delta_evento, 0.0)
            pred_delta_dummy_gate = np.where(pred_evento == 1, reg_dummy_evento.predict(Xv_evento[idx]), 0.0)
            pred_delta_zero = np.zeros(len(idx), dtype=float)

            linhas_regressao.extend([
                avaliar_delta_unico(y_delta_v[idx], pred_delta_gate, "Pipeline completo", nome_split),
                avaliar_delta_unico(y_delta_v[idx], pred_delta_evento_real_gate, "Evento real + regressor (diagnostico)", nome_split),
                avaliar_delta_unico(y_delta_v[idx], pred_delta_dummy_gate, "Classificador + mediana", nome_split),
                avaliar_delta_unico(y_delta_v[idx], pred_delta_zero, "Sempre prever zero", nome_split),
            ])

            predicoes_evento[nome_split] = {
                "idx": idx,
                "proba_evento": proba_evento,
                "evento_real": y_evento_v[idx],
                "evento_pred": pred_evento,
                "delta_real": y_delta_v[idx],
                "delta_pred_gate": pred_delta_gate,
                "delta_pred_evento_real_gate": pred_delta_evento_real_gate,
            }

        resultados_classificacao_evento = pd.DataFrame(linhas_classificacao)
        resultados_regressao_evento = pd.DataFrame(linhas_regressao)

        display(Markdown(
            "### Etapa 1 - detectar se houve mudanca relevante de proximidade\n"
            "A classe majoritaria preve evento em todos os intervalos. Por isso ela tem recall 1, "
            "mas balanced accuracy 0,5: encontrar todos os eventos nao significa separar bem as duas classes."
        ))
        display(resultados_classificacao_evento.round(3))

        metricas_classificador = (
            resultados_classificacao_evento
            .query("modelo == 'MLP evento'")
            .melt(
                id_vars=["split"],
                value_vars=["balanced_acc", "precision_evento", "recall_evento", "f1_evento"],
                var_name="metrica",
                value_name="valor",
            )
        )
        metricas_classificador["metrica"] = metricas_classificador["metrica"].map({
            "balanced_acc": "Balanced accuracy",
            "precision_evento": "Precisao",
            "recall_evento": "Recall",
            "f1_evento": "F1",
        })
        fig_metricas_evento = px.bar(
            metricas_classificador,
            x="metrica",
            y="valor",
            color="split",
            barmode="group",
            text="valor",
            range_y=[0, 1.05],
            title="Desempenho do classificador de eventos",
            labels={"metrica": "Metrica", "valor": "Resultado", "split": "Conjunto"},
            template="plotly_white",
        )
        fig_metricas_evento.update_traces(texttemplate="%{text:.3f}", textposition="outside")
        fig_metricas_evento.show()

        resultado_clf_teste = resultados_classificacao_evento.query(
            "modelo == 'MLP evento' and split == 'teste'"
        ).iloc[0]
        matriz_teste = np.array([
            [resultado_clf_teste["tn"], resultado_clf_teste["fp"]],
            [resultado_clf_teste["fn"], resultado_clf_teste["tp"]],
        ], dtype=int)
        fig_confusao = go.Figure(go.Heatmap(
            z=matriz_teste,
            x=["Predito: sem evento", "Predito: evento"],
            y=["Real: sem evento", "Real: evento"],
            text=matriz_teste,
            texttemplate="%{text}",
            colorscale="Blues",
            showscale=False,
        ))
        fig_confusao.update_layout(
            title="Onde o classificador acertou e errou no teste fixo",
            template="plotly_white",
            yaxis_autorange="reversed",
        )
        fig_confusao.show()

        display(Markdown(
            "### Etapa 2 - estimar o tamanho da mudanca\n"
            f"O regressor de magnitude usa dropout de {reg_evento.dropout:.0%} e early stopping. "
            f"A melhor epoca foi {reg_evento.best_epoch_}, com interrupcao na epoca "
            f"{reg_evento.stopped_epoch_}.\n\n"
            "`Evento real + regressor (diagnostico)` informa ao regressor quais intervalos realmente "
            "tinham evento. Essa opcao nao pode ser usada em producao; ela serve apenas para verificar "
            "se o erro vem principalmente do classificador ou do regressor."
        ))
        resultados_regressao_teste = (
            resultados_regressao_evento
            .query("split == 'teste'")
            .sort_values("MAE")
            .reset_index(drop=True)
        )
        display(resultados_regressao_teste.round(3))
        fig_mae_teste = px.bar(
            resultados_regressao_teste,
            x="modelo",
            y="MAE",
            color="modelo",
            text="MAE",
            title=f"Erro final no teste fixo para {EVENT_TARGET_DELTA}",
            labels={"modelo": "Estrategia", "MAE": "MAE (p.p.)"},
            template="plotly_white",
        )
        fig_mae_teste.update_traces(texttemplate="%{text:.3f}", textposition="outside")
        fig_mae_teste.update_layout(showlegend=False)
        fig_mae_teste.show()

        mae_pipeline = resultados_regressao_teste.loc[
            resultados_regressao_teste["modelo"] == "Pipeline completo", "MAE"
        ].iloc[0]
        mae_evento_real = resultados_regressao_teste.loc[
            resultados_regressao_teste["modelo"] == "Evento real + regressor (diagnostico)", "MAE"
        ].iloc[0]
        variacao_evento_real_pct = (mae_evento_real / max(mae_pipeline, 1e-9) - 1.0) * 100.0
        if abs(variacao_evento_real_pct) <= 5.0:
            leitura_diagnostico = (
                "Os valores ficaram proximos, entao os erros do classificador nao explicam a maior "
                "parte do erro final."
            )
        elif variacao_evento_real_pct < 0.0:
            leitura_diagnostico = (
                "O MAE diminuiu com o evento real, indicando que os erros do classificador contribuem "
                "para o erro final."
            )
        else:
            leitura_diagnostico = (
                "O MAE aumentou mesmo com o evento real. Isso nao indica ganho do classificador; mostra "
                "que o regressor de magnitude ainda esta instavel nos intervalos de evento."
            )
        display(Markdown(
            f"**Leitura direta:** no teste, fornecer o evento real ao regressor mudou o MAE de "
            f"{mae_pipeline:.3f} para {mae_evento_real:.3f} p.p. {leitura_diagnostico}"
        ))

        maior_mae_validacao = resultados_regressao_evento.query("split == 'validacao'")["MAE"].max()
        maior_mae_teste = resultados_regressao_teste["MAE"].max()
        if maior_mae_validacao > max(100.0, maior_mae_teste * 10.0):
            display(Markdown(
                f"**Alerta de instabilidade:** o maior MAE da validacao foi {maior_mae_validacao:.3e} p.p. "
                "Esse valor extremo achatava as demais barras no grafico anterior. O grafico principal acima "
                "mostra somente o teste fixo, mas a divergencia da validacao continua registrada e indica "
                "que o regressor desta segunda etapa ainda nao esta estavel."
            ))

        if "teste" in predicoes_evento:
            pred_teste_evento = predicoes_evento["teste"]
            meta_teste_evento = meta_evento.iloc[pred_teste_evento["idx"]].reset_index(drop=True)
            inspecao_evento = pd.DataFrame({
                "run_id": meta_teste_evento["run_id"],
                "sample_id": meta_teste_evento["sample_id"],
                "delta_real": pred_teste_evento["delta_real"],
                "evento_real": pred_teste_evento["evento_real"],
                "proba_evento": pred_teste_evento["proba_evento"],
                "evento_pred": pred_teste_evento["evento_pred"],
                "delta_pred_gate": pred_teste_evento["delta_pred_gate"],
                "delta_pred_evento_real_gate": pred_teste_evento["delta_pred_evento_real_gate"],
            })
            display(Markdown("**Intervalos de teste mais extremos para inspecao:**"))
            display(inspecao_evento.assign(abs_delta=lambda df: df["delta_real"].abs()).sort_values("abs_delta", ascending=False).drop(columns="abs_delta").head(20))



c:\Users\Prograf_4080Super\.conda\envs\tcc-drone-metrics\Lib\site-packages\sklearn\metrics\_classification.py:614: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.
  warnings.warn(


**Curva do classificador com validacao e teste fixos:**

,modelo,split,event_rate_real,event_rate_pred,balanced_acc,precision_evento,recall_evento,f1_evento,avg_precision,tn,fp,fn,tp,marco_runs,estrategia_split
0,MLP evento,teste,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,NaN,0,0,0,1,9,validacao_teste_fixos
1,MLP evento,teste,0.914286,0.907143,0.585938,0.929134,0.921875,0.925490,0.932434,3,9,10,118,17,validacao_teste_fixos
2,MLP evento,teste,0.900943,0.879717,0.684306,0.938338,0.916230,0.927152,0.955926,19,23,32,350,25,validacao_teste_fixos
3,MLP evento,teste,0.910369,0.920914,0.758101,0.956107,0.967181,0.961612,0.974964,28,23,17,501,33,validacao_teste_fixos
4,MLP evento,teste,0.909686,0.941099,0.690668,0.942976,0.975540,0.958982,0.969936,28,41,17,678,50,validacao_teste_fixos
5,MLP evento,teste,0.891827,0.914663,0.726250,0.939553,0.963612,0.951430,0.955848,44,46,27,715,67,validacao_teste_fixos
6,MLP evento,teste,0.908293,0.904390,0.740179,0.952535,0.948443,0.950484,0.965838,50,44,48,883,85,validacao_teste_fixos
7,MLP evento,teste,0.910665,0.900638,0.764790,0.958502,0.947948,0.953196,0.965556,57,41,52,947,102,validacao_teste_fixos
8,MLP evento,validacao,0.922414,0.931034,0.703531,0.953704,0.962617,0.958140,0.956316,4,5,4,103,9,validacao_teste_fixos
9,MLP evento,validacao,0.877193,0.789474,0.868571,0.977778,0.880000,0.926316,0.960372,6,1,6,44,17,validacao_teste_fixos


**Dataset evento proximidade:** 8298 intervalos, 115 features com contexto temporal, alvo `delta_depth_close_5m_pp`, limiar=0.50 p.p. Taxa de evento=90.2%. Separacao: temporal por runs completas. Treino=6043, validacao=1158, teste=1097.

,run_id,intervalos,eventos,taxa_evento,delta_abs_p90
0,run_20260819_164000_020945,33,22,0.666667,19.141900
1,run_20260819_164322_923441,21,14,0.666667,13.498394
2,run_20260819_164554_301264,24,16,0.666667,13.805578
3,run_20260819_171344_684737,24,16,0.666667,10.245236
4,run_20260819_171520_281146,139,127,0.913669,20.534835
...,...,...,...,...,...
97,run_20260831_083731_750398,72,66,0.916667,25.438721
98,run_20260831_083939_468039,67,61,0.910448,23.532924
99,run_20260831_084145_348727,72,67,0.930556,19.733971
100,run_20260831_084356_708163,67,61,0.910448,23.544885


### Etapa 1 - detectar se houve mudanca relevante de proximidade
A classe majoritaria preve evento em todos os intervalos. Por isso ela tem recall 1, mas balanced accuracy 0,5: encontrar todos os eventos nao significa separar bem as duas classes.

... (2 outputs left out)

,modelo,split,alvo_delta,MAE,RMSE,corr
0,Evento real + regressor (diagnostico),teste,delta_depth_close_5m_pp,6.179,9.023,0.725
1,Pipeline completo,teste,delta_depth_close_5m_pp,6.292,9.081,0.721
2,Classificador + mediana,teste,delta_depth_close_5m_pp,9.510,13.038,0.009
3,Sempre prever zero,teste,delta_depth_close_5m_pp,9.511,13.040,NaN


**Leitura direta:** no teste, fornecer o evento real ao regressor mudou o MAE de 6.292 para 6.179 p.p. Os valores ficaram proximos, entao os erros do classificador nao explicam a maior parte do erro final.

**Intervalos de teste mais extremos para inspecao:**

,run_id,sample_id,delta_real,evento_real,proba_evento,evento_pred,delta_pred_gate,delta_pred_evento_real_gate
781,run_20260831_083731_750398,24,50.281707,1,1.0,1,25.724743,25.724743
105,run_20260831_081551_447997,22,47.416070,1,1.0,1,32.014453,32.014453
106,run_20260831_081551_447997,23,-45.528552,1,1.0,1,-15.141028,-15.141028
769,run_20260831_083731_750398,12,43.294852,1,1.0,1,65.402140,65.402140
194,run_20260831_081809_010410,33,38.002544,1,1.0,1,24.961776,24.961776
258,run_20260831_082026_776169,18,37.649545,1,1.0,1,16.980795,16.980795
198,run_20260831_081809_010410,37,36.973557,1,1.0,1,0.047509,0.047509
393,run_20260831_082629_864845,14,36.124402,1,1.0,1,14.937719,14.937719
473,run_20260831_082836_332335,20,-36.106821,1,1.0,1,-9.661038,-9.661038
398,run_20260831_082629_864845,19,-36.087475,1,1.0,1,-11.831230,-11.831230


### Validacao da sincronizacao entre flow, IMU e depth

A pergunta principal Ã© se os intervalos salvos em memmap estao coerentes: `dt_s`, `depth_age_s`, `depth_dt_s`, vetores de optical flow e deltas de depth devem descrever a mesma janela temporal da logica de proximidade visual.

In [18]:
COLUNAS_SYNC_INTERVALOS = ["dt_s", "depth_age_s", "depth_dt_s", "flow_valid_points", "flow_track_retention_pct", "flow_mag_p90_px", "radial_flow_p90_px", "delta_depth_close_5m_pp", "delta_x_m", "delta_y_m", "delta_yaw_heading_rad", "pan_comp_delta_rad"]


def montar_dataframe_intervalos_memmap(base_dir):
    """Concatena os intervalos sincronizados de todas as runs depth."""

    runs = carregar_todas_runs_depth_memmap(base_dir)
    if not runs:
        return pd.DataFrame()
    partes = []
    for run in runs:
        df = run["intervalos"].copy()
        df["run_id"] = run["run_dir"].name
        partes.append(df)
    return pd.concat(partes, ignore_index=True) if partes else pd.DataFrame()


def resumir_sincronizacao_intervalos(df):
    if df.empty:
        return pd.DataFrame()
    cols = [c for c in COLUNAS_SYNC_INTERVALOS if c in df.columns]
    return df[cols].describe().T


raiz_flow = localizar_raiz_projeto_memmap()
intervalos_flow_df = montar_dataframe_intervalos_memmap(raiz_flow)

if intervalos_flow_df.empty:
    display(Markdown("Nenhum intervalo memmap encontrado para validar sincronizacao flow/depth."))
else:
    print(f"Intervalos avaliados: {len(intervalos_flow_df)}")
    print(intervalos_flow_df.groupby("run_id").size().rename("intervalos"))
    display(resumir_sincronizacao_intervalos(intervalos_flow_df))
    px.box(intervalos_flow_df, x="run_id", y="dt_s", title="Distribuicao do intervalo temporal entre atualizacoes visuais", labels={"dt_s": "Delta de tempo do intervalo visual (s)", "run_id": "Run"}, template="plotly_white").show()
    px.scatter(intervalos_flow_df, x="radial_flow_p90_px", y="delta_depth_close_5m_pp", color="run_id", size="flow_valid_points", hover_data=["sample_id", "dt_s", "depth_age_s", "pan_comp_delta_rad"], title="Flow radial do intervalo x variacao de proximidade no depth", labels={"radial_flow_p90_px": "P90 do flow radial (px)", "delta_depth_close_5m_pp": "Delta pixels < 5 m (p.p.)"}, template="plotly_white").show()
    serie = intervalos_flow_df.reset_index(drop=True)
    fig_tempo = go.Figure()
    fig_tempo.add_trace(go.Scatter(x=serie.index, y=serie["flow_mag_p90_px"], mode="lines", name="P90 flow"))
    fig_tempo.add_trace(go.Scatter(x=serie.index, y=serie["delta_depth_close_5m_pp"], mode="lines", name="Delta pixels < 5m", yaxis="y2"))
    fig_tempo.update_layout(title="Sequencia dos intervalos: flow visual e delta de proximidade", xaxis_title="Intervalos concatenados em ordem de leitura", yaxis=dict(title="P90 flow (px)"), yaxis2=dict(title="Delta pixels < 5m (p.p.)", overlaying="y", side="right"), template="plotly_white", hovermode="x unified")
    fig_tempo.show()

Intervalos avaliados: 8305
run_id
run_20260819_164000_020945     33
run_20260819_164322_923441     21
run_20260819_164554_301264     24
run_20260819_171344_684737     24
run_20260819_171520_281146    139
                             ... 
run_20260831_083731_750398     72
run_20260831_083939_468039     67
run_20260831_084145_348727     72
run_20260831_084356_708163     67
run_20260831_214900_679028     61
Name: intervalos, Length: 103, dtype: int64


,count,mean,std,min,25%,50%,75%,max
dt_s,8305.0,1.017537,0.839101,0.328000,0.596000,0.760000,1.092000,9.736000
depth_age_s,8305.0,0.000421,0.013901,0.000000,0.000000,0.000000,0.000000,0.592000
depth_dt_s,8305.0,1.017537,0.839101,0.328000,0.596000,0.760000,1.092000,9.736000
flow_valid_points,8305.0,45.045274,20.526215,0.000000,31.000000,41.000000,56.000000,155.000000
flow_track_retention_pct,8305.0,65.790065,21.099228,0.000000,50.704225,67.592593,82.352941,100.000000
flow_mag_p90_px,8305.0,80.906530,50.509397,0.000000,43.513632,75.338787,112.372667,281.018335
radial_flow_p90_px,8305.0,23.363340,25.904491,-217.103813,7.194924,22.661497,38.917307,119.483218
delta_depth_close_5m_pp,8298.0,0.309773,11.915115,-58.525836,-6.342729,0.037262,6.583080,51.596781
delta_x_m,8305.0,-0.258422,1.470836,-21.405968,-0.925186,-0.090607,0.326298,15.023520
delta_y_m,8305.0,0.249656,1.944246,-19.564867,-0.548658,0.258318,1.278610,21.037645


### Experimentos complementares: sementes, arvores e ablacao

Esta etapa usa o conjunto completo e preserva as execucoes fixas de validacao e teste. A MLP e treinada com cinco sementes para medir a variacao das metricas. Random Forest e Extra Trees formam o baseline tabular com arvores. Na ablacao, um grupo de entradas e removido por vez; aumento positivo do MAE indica que a retirada daquele grupo prejudicou o modelo.

O bloco executa cinco treinamentos de robustez, dois modelos de arvores e tres treinamentos por configuracao de ablacao. Os resultados detalhados e resumidos sao salvos em CSV dentro de `estudos_e_analises`.

In [19]:
if not all(nome in globals() for nome in (
    "X_mlp_delta_explicacao",
    "y_mlp_delta_explicacao",
    "meta_mlp_delta_explicacao",
    "split_mlp_delta_explicacao",
    "classificar_grupo_feature",
)):
    display(Markdown("Execute primeiro as celulas da MLP, da representacao angular e da explicabilidade."))
else:
    raiz_experimentos = localizar_raiz_projeto_memmap()
    diretorio_analises = raiz_experimentos / "estudos_e_analises"
    diretorio_logs_analises = diretorio_analises / "logs"
    diretorio_logs_analises.mkdir(parents=True, exist_ok=True)

    SEMENTES_ROBUSTEZ = (7, 21, 42, 84, 123)
    X_experimentos = X_mlp_delta_explicacao.copy()
    y_experimentos = y_mlp_delta_explicacao.copy()
    meta_experimentos = meta_mlp_delta_explicacao.copy()
    split_experimentos = split_mlp_delta_explicacao

    def criar_mlp_experimento(semente):
        configuracao = {**REGRESSOR_CONFIG, "random_state": int(semente)}
        return EarlyStoppingDropoutRegressor(**configuracao)

    resultados_sementes, resumo_sementes = executar_multiplas_sementes(
        X_experimentos,
        y_experimentos,
        split_experimentos,
        criar_mlp_experimento,
        SEMENTES_ROBUSTEZ,
    )
    resultados_sementes.to_csv(diretorio_logs_analises / "robustez_multiplas_sementes.csv", index=False)
    resumo_sementes.to_csv(diretorio_logs_analises / "robustez_multiplas_sementes_resumo.csv", index=False)

    display(Markdown("**Robustez da MLP a diferentes sementes**"))
    display(resumo_sementes.round(4))
    px.box(
        resultados_sementes.query("split == 'teste'"),
        x="alvo_delta",
        y="MAE",
        points="all",
        hover_data=["semente", "melhor_epoca"],
        title="Variacao do MAE no teste fixo entre sementes",
        labels={"alvo_delta": "Alvo", "MAE": "MAE"},
        template="plotly_white",
    ).show()

    resultados_arvores, modelos_arvores = executar_baselines_arvores(
        X_experimentos, y_experimentos, split_experimentos, random_state=42
    )
    X_experimentos_array = X_experimentos.replace([np.inf, -np.inf], np.nan).fillna(0.0).to_numpy(float)
    y_experimentos_array = y_experimentos.replace([np.inf, -np.inf], np.nan).fillna(0.0).to_numpy(float)
    resultados_mlp_referencia = []
    for nome_split, chave_split in (("validacao", "val"), ("teste", "test")):
        indices = split_experimentos[chave_split]
        predicao = modelo_mlp_delta_explicacao.predict(X_experimentos_array[indices])
        resultados_mlp_referencia.extend(avaliar_delta(
            y_experimentos_array[indices], predicao, y_experimentos.columns, "MLP", nome_split
        ))
    comparacao_arvores = pd.concat(
        [resultados_arvores, pd.DataFrame(resultados_mlp_referencia)], ignore_index=True
    )
    comparacao_arvores.to_csv(diretorio_logs_analises / "baseline_arvores.csv", index=False)

    display(Markdown("**MLP versus modelos de arvores e media do treino**"))
    display(comparacao_arvores.query("split == 'teste'").sort_values(["alvo_delta", "MAE"]).round(4))
    px.bar(
        comparacao_arvores.query("split == 'teste'"),
        x="alvo_delta",
        y="MAE",
        color="modelo",
        barmode="group",
        title="Baseline tabular no teste fixo",
        labels={"alvo_delta": "Alvo", "MAE": "MAE", "modelo": "Modelo"},
        template="plotly_white",
    ).show()

    resultados_ablacao, comparacao_ablacao = executar_ablacao_grupos(
        X_experimentos,
        y_experimentos,
        split_experimentos,
        criar_mlp_experimento,
        classificar_grupo_feature,
        sementes=(21, 42, 84),
    )
    resultados_ablacao.to_csv(diretorio_logs_analises / "ablacao_grupos_entradas.csv", index=False)
    comparacao_ablacao.to_csv(diretorio_logs_analises / "ablacao_grupos_entradas_comparacao.csv", index=False)

    ablacao_teste = comparacao_ablacao.query("split == 'teste' and grupo_removido != 'Nenhum'")
    ablacao_teste_resumo = (
        ablacao_teste.groupby(["alvo_delta", "grupo_removido"], as_index=False)
        .agg(
            delta_MAE_pct_medio=("delta_MAE_pct", "mean"),
            delta_MAE_pct_desvio=("delta_MAE_pct", "std"),
            delta_MAE_pct_minimo=("delta_MAE_pct", "min"),
            delta_MAE_pct_maximo=("delta_MAE_pct", "max"),
        )
    )
    ablacao_teste_resumo.to_csv(diretorio_logs_analises / "ablacao_grupos_entradas_resumo.csv", index=False)
    display(Markdown("**Ablacao por retirada de grupos: valor positivo indica piora sem o grupo**"))
    display(ablacao_teste_resumo.sort_values(["alvo_delta", "delta_MAE_pct_medio"], ascending=[True, False]).round(3))
    fig_ablacao = px.bar(
        ablacao_teste_resumo,
        x="grupo_removido",
        y="delta_MAE_pct_medio",
        error_y="delta_MAE_pct_desvio",
        facet_col="alvo_delta",
        facet_col_wrap=3,
        title="Ablacao no teste fixo: variacao do MAE ao remover cada grupo",
        labels={"grupo_removido": "Grupo removido", "delta_MAE_pct_medio": "Variacao media do MAE (%)"},
        template="plotly_white",
    )
    fig_ablacao.add_hline(y=0, line_dash="dash")
    fig_ablacao.update_xaxes(tickangle=35)
    fig_ablacao.show()

**Robustez da MLP a diferentes sementes**

,split,alvo_delta,MAE_medio,MAE_desvio,MAE_minimo,MAE_maximo,RMSE_medio,RMSE_desvio,corr_media,melhor_epoca_media,MAE_ic95
0,teste,delta_depth_close_10m_pp,3.0550,0.0189,3.0245,3.0744,4.5156,0.0304,0.7830,162.8,0.0166
1,teste,delta_depth_close_2m_pp,3.5166,0.0691,3.4300,3.5841,5.0796,0.0960,0.8410,162.8,0.0605
2,teste,delta_depth_close_5m_pp,5.2682,0.0681,5.1953,5.3663,7.2928,0.1006,0.8350,162.8,0.0597
3,teste,delta_depth_p10_m,0.3639,0.0023,0.3606,0.3670,0.5296,0.0063,0.8002,162.8,0.0020
4,teste,delta_depth_p50_m,0.8114,0.0047,0.8052,0.8180,1.3004,0.0106,0.8339,162.8,0.0042
5,teste,delta_depth_p90_m,2.9859,0.0534,2.9349,3.0680,4.7879,0.0855,0.7360,162.8,0.0468
6,validacao,delta_depth_close_10m_pp,3.3188,0.0232,3.2835,3.3479,4.9629,0.0457,0.7483,162.8,0.0204
7,validacao,delta_depth_close_2m_pp,2.9090,0.0485,2.8310,2.9553,4.3078,0.0685,0.8273,162.8,0.0425
8,validacao,delta_depth_close_5m_pp,5.3592,0.0412,5.3145,5.4214,7.3903,0.0375,0.8127,162.8,0.0361
9,validacao,delta_depth_p10_m,0.3606,0.0020,0.3580,0.3628,0.5506,0.0046,0.7956,162.8,0.0018


**MLP versus modelos de arvores e media do treino**

,modelo,split,alvo_delta,MAE,RMSE,corr
35,Extra Trees,teste,delta_depth_close_10m_pp,2.9517,4.4865,0.7854
23,Random Forest,teste,delta_depth_close_10m_pp,2.9602,4.5123,0.7833
47,MLP,teste,delta_depth_close_10m_pp,3.0660,4.5038,0.7813
11,Media treino,teste,delta_depth_close_10m_pp,4.5284,7.1307,NaN
33,Extra Trees,teste,delta_depth_close_2m_pp,3.2031,4.6817,0.8589
21,Random Forest,teste,delta_depth_close_2m_pp,3.2622,4.7480,0.8529
45,MLP,teste,delta_depth_close_2m_pp,3.4571,4.9933,0.8430
9,Media treino,teste,delta_depth_close_2m_pp,5.6288,8.7234,NaN
22,Random Forest,teste,delta_depth_close_5m_pp,5.0333,6.9982,0.8457
34,Extra Trees,teste,delta_depth_close_5m_pp,5.0396,7.0129,0.8445


**Ablacao por retirada de grupos: valor positivo indica piora sem o grupo**

,alvo_delta,grupo_removido,delta_MAE_pct_medio,delta_MAE_pct_desvio,delta_MAE_pct_minimo,delta_MAE_pct_maximo
1,delta_depth_close_10m_pp,Diferenca de imagem,13.667,0.923,12.603,14.259
2,delta_depth_close_10m_pp,IMU e atitude,8.546,2.547,6.580,11.424
3,delta_depth_close_10m_pp,Movimento,3.917,0.351,3.511,4.120
4,delta_depth_close_10m_pp,Optical flow e risco,1.291,1.230,0.315,2.673
0,delta_depth_close_10m_pp,Comandos reativos,1.165,0.680,0.715,1.948
5,delta_depth_close_10m_pp,Sincronizacao,0.704,1.530,-0.579,2.397
9,delta_depth_close_2m_pp,Movimento,7.814,0.914,7.010,8.809
8,delta_depth_close_2m_pp,IMU e atitude,4.117,0.442,3.631,4.496
10,delta_depth_close_2m_pp,Optical flow e risco,2.463,2.129,0.659,4.812
7,delta_depth_close_2m_pp,Diferenca de imagem,0.482,1.332,-0.898,1.760


### Mapeamento espacial 3D, A* e gate monocular final

Esta seção lê os artefatos consolidados em logs/spatial_final_analysis/final_* e as trajetórias gravadas em datasets/spatial_mapping/run_*. Os gráficos separam a bateria com profundidade de referência da avaliação monocular v12--v19.

In [20]:
import sys
from pathlib import Path
from IPython.display import Markdown, display

candidatos_modulo_espacial = [
    Path.cwd(),
    Path.cwd() / "estudos_e_analises",
    Path.cwd().parent / "estudos_e_analises",
]
for candidato in candidatos_modulo_espacial:
    if (candidato / "spatial_metrics.py").exists():
        sys.path.insert(0, str(candidato))
        break

from spatial_metrics import (
    load_spatial_results,
    monocular_models_figure,
    reference_runs_figure,
    reference_trajectories_figure,
)

relatorio_espacial, runs_referencia, modelos_monoculares = load_spatial_results()
resumo_referencia = relatorio_espacial.get("reference", {})
display(Markdown(
    f"**Referência:** {resumo_referencia.get('completed_runs', 0)}/"
    f"{resumo_referencia.get('runs', 0)} runs completas; "
    f"{100 * resumo_referencia.get('plan_success_rate', 0):.1f}% de sucesso "
    f"dos planos; {resumo_referencia.get('unsafe_adopted_paths', 0)} caminhos "
    "inseguros adotados. **Monocular:** bloqueado pelo gate espacial final."
))

reference_runs_figure(runs_referencia).show()
reference_trajectories_figure(runs_referencia).show()
monocular_models_figure(modelos_monoculares).show()

**Referência:** 10/10 runs completas; 94.2% de sucesso dos planos; 0 caminhos inseguros adotados. **Monocular:** bloqueado pelo gate espacial final.

## Encerramento monocular: versão final
Validação de desenvolvimento: 77 quadros, 17 pedidos. Não são voos nem teste independente. A célula abaixo regenera as figuras a partir dos resultados versionados, sem retreinar.

In [ ]:
from pathlib import Path
import json, subprocess, sys
root = Path.cwd()
if not (root / "scripts/reproduce_final_results.py").exists(): root = root.parent
subprocess.run([sys.executable, str(root / "scripts/reproduce_final_results.py")], check=True)
final = json.loads((root / "datasets/resultados_finais_monocular/summary.json").read_text())
final
